# 사내 복리후생·인사 RAG 챗봇 — 베이스라인 (v2: BGE-M3 + 규칙)

**설계 원칙**
1. **LLM은 답변 생성에만** 씁니다.
2. 나머지(질문 재작성 · 분류 · 라우팅 · 검색 · 재정렬)는 전부 **BGE-M3 + 규칙(사전·정규식·템플릿)** 으로 처리합니다.

| 단계 | 하는 일 | 담당 |
|---|---|---|
| ① 질문 재작성 | 줄임말 치환, 규정 용어 덧붙이기, 이전 대화의 이벤트·조건 이어받기, 복합 질문 분리 | 규칙 |
| ② 질문 분류 | 업무 영역(멀티라벨) · 이벤트 · 조건(slot) · 범위 밖 · 개인 조회 · 민감 · 애매함 | BGE-M3 프로토타입 + 규칙 |
| ③ 라우터 | 민감 → 담당자 / 범위 밖 → 거절 / 개인 조회 → HR 시스템 / 애매 → 되묻기 / 나머지 → 검색 | 규칙 |
| ④ 검색기 | (필터) → Dense + Sparse → RRF → ColBERT 재정렬 → 조항 Top-5 | BGE-M3 |
| ⑤ 답변 | Top-5 근거로만 답하고 출처(몇 조) 표시 | **LLM (유일)** |
| + 실패 처리 | 점수가 낮으면 확장 재검색 1회 → 그래도 낮으면 "답변 불가" + 추천 질문 3개 | BGE-M3 + 규칙 |

**비교 실험 (같은 평가셋, 같은 청크)** — A0 노트북과 청킹·지표가 같아서 점수를 그대로 이어 붙일 수 있습니다.

| 구성 | 추가되는 것 |
|---|---|
| A0 | Dense만 (A0 노트북과 같은 조건) |
| A1 | + BGE-M3 Sparse, RRF로 합치기 |
| A1k | (선택) Sparse 대신 BM25 + Kiwi |
| A2 | + ColBERT 재정렬 (BGE-M3 dense·sparse·colbert 가중합) |
| A3 | + 질문 재작성 (멀티턴·줄임말·복합 질문) |
| B | + 분류 결과로 하드 필터 |
| **C** | + confidence에 따라 하드/소프트 필터, 약하면 필터 해제 (**최종 베이스라인**) |

`data/regulations/`가 비어 있으면 **가상 샘플**로 끝까지 돌아갑니다. 샘플 점수는 코드 점검용이고 보고용이 아닙니다.

## 0. 설치 · 설정

In [29]:
!pip install -q -U FlagEmbedding kiwipiepy pdfplumber pandas scikit-learn openai

In [30]:
import os, re, json, time, zipfile, hashlib, subprocess, shutil, platform, pickle
import xml.etree.ElementTree as ET
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
from IPython.display import display

CONFIG = {
    "DATA_DIR": "data/regulations",
    "EVAL_PATH": "data/eval_set.csv",
    "RESULTS_DIR": "results",
    "CACHE_DIR": "cache",
    # "bge-m3" = 실제 베이스라인 / "mock" = 모델 다운로드 없이 코드 흐름만 점검
    "EMBEDDER": os.environ.get("BASELINE_EMBEDDER", "bge-m3"),
    "MODEL_NAME": "BAAI/bge-m3",
    "MAX_SEQ_LEN": 1024,        # 메모리 부족하면 512
    "BATCH_SIZE": 8,
    "CHUNK_MAX_CHARS": 1200,    # A0와 같게 유지해야 점수 비교가 됩니다
    "N_CANDIDATES": 50,         # Dense·Sparse 각각 뽑는 후보 수
    "RRF_K": 60,
    "RERANK_POOL": 20,          # ColBERT로 다시 점수 매길 청크 수
    "TOP_K": 5,                 # LLM에 넘길 조항 수
    "EVAL_K": 10,               # 평가할 때 보는 조항 수
    "M3_WEIGHTS": {"dense": 0.4, "sparse": 0.2, "colbert": 0.4},  # BGE-M3 공식 권장 비율
    "MIN_FILTERED": 3,          # 하드 필터 뒤 남은 후보가 이보다 적으면 필터 해제
    "SOFT_BOOST": 0.3,          # confidence low일 때 카테고리가 맞는 청크의 RRF 점수 가산 비율
    "PROTO_MARGIN": 0.03,       # 1등 카테고리와 이만큼 가까우면 같이 붙임 (멀티라벨)
    "AUTO_CALIBRATE": True,     # 평가셋으로 범위밖·답변불가 임계값을 자동으로 정함
    "EVAL_INTENTS": ["규정", "둘다", "이벤트"],   # 검색 평가 대상 intent
    "SEED": 42,
}
# 임계값 기본값. AUTO_CALIBRATE가 켜져 있으면 9절에서 평가셋으로 다시 정합니다.
TH_DEFAULTS = {"bge-m3": {"in_scope": 0.40, "high_conf": 0.55, "answer": 0.40},
               "mock":   {"in_scope": 0.10, "high_conf": 0.20, "answer": 0.15}}
TH = dict(TH_DEFAULTS[CONFIG["EMBEDDER"]])
for d in (CONFIG["DATA_DIR"], CONFIG["RESULTS_DIR"], CONFIG["CACHE_DIR"]):
    Path(d).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=1))

{
 "DATA_DIR": "data/regulations",
 "EVAL_PATH": "data/eval_set.csv",
 "RESULTS_DIR": "results",
 "CACHE_DIR": "cache",
 "EMBEDDER": "bge-m3",
 "MODEL_NAME": "BAAI/bge-m3",
 "MAX_SEQ_LEN": 1024,
 "BATCH_SIZE": 8,
 "CHUNK_MAX_CHARS": 1200,
 "N_CANDIDATES": 50,
 "RRF_K": 60,
 "RERANK_POOL": 20,
 "TOP_K": 5,
 "EVAL_K": 10,
 "M3_WEIGHTS": {
  "dense": 0.4,
  "sparse": 0.2,
  "colbert": 0.4
 },
 "MIN_FILTERED": 3,
 "SOFT_BOOST": 0.3,
 "PROTO_MARGIN": 0.03,
 "AUTO_CALIBRATE": true,
 "EVAL_INTENTS": [
  "규정",
  "둘다",
  "이벤트"
 ],
 "SEED": 42
}


## 1. 데이터
### (자동) 가상 샘플
`data/regulations/`에 실제 파일이 없을 때만 샘플 규정을 만들고, 평가셋도 `data/SAMPLE_eval_set.csv`를 씁니다. 실제 `data/eval_set.csv`는 건드리지 않습니다.

평가셋 컬럼은 A0와 같고, **선택 컬럼 2개**가 추가됐습니다.
- `history`: 멀티턴 질문의 이전 사용자 발화. 여러 개면 ` || `로 구분
- `expected_route`: 정답 라우트 (`검색` / `개인조회` / `되묻기` / `범위밖` / `담당자연결`). 비우면 intent로 정합니다 (민감 → 담당자연결, 내정보 → 개인조회, 범위밖 → 범위밖, 나머지 → 검색)

In [31]:
SAMPLE_DOCS = {
"SAMPLE_누리정밀_취업규칙.txt": """누리정밀 취업규칙
제1장 총칙
제1조(목적) 이 규칙은 누리정밀주식회사(이하 "회사"라 한다) 직원의 근로조건과 복무에 관한 사항을 정함을 목적으로 한다.
제2조(적용범위) 이 규칙은 회사에 근무하는 모든 직원에게 적용한다. 다만, 다른 규정에 특별한 정함이 있는 경우에는 그에 따른다.
제4장 휴일 및 휴가
제25조(연차유급휴가) ① 회사는 1년간 80퍼센트 이상 출근한 직원에게 15일의 유급휴가를 준다.
② 계속하여 근로한 기간이 1년 미만인 직원에게는 1개월 개근 시 1일의 유급휴가를 준다.
③ 3년 이상 계속 근로한 직원에게는 최초 1년을 초과하는 계속 근로연수 매 2년에 대하여 1일을 가산한다. 이 경우 가산휴가를 포함한 총 휴가일수는 25일을 한도로 한다.
④ 연차유급휴가는 반일 단위로 나누어 사용할 수 있다. 오전 반차는 09:00부터 13:00까지, 오후 반차는 14:00부터 18:00까지로 한다.
⑤ 사용하지 아니한 연차유급휴가는 제25조의2에 따라 보상한다.
제25조의2(연차휴가 미사용 수당) 회사는 제25조에 따른 휴가를 사용하지 못한 직원에게 미사용 일수에 대하여 통상임금을 기준으로 연차휴가 미사용 수당을 지급한다. 다만, 회사가 근로기준법 제61조에 따라 사용을 촉진한 경우에는 그러하지 아니하다.
제26조(경조휴가) ① 회사는 다음 각 호의 경조사가 있는 직원에게 다음 표의 경조휴가를 준다.
| 구분 | 대상 | 휴가일수 |
|---|---|---|
| 결혼 | 본인 | 5일 |
| 결혼 | 자녀 | 1일 |
| 출산 | 배우자 | 10일 |
| 사망 | 배우자, 본인 및 배우자의 부모 | 5일 |
| 사망 | 본인 및 배우자의 조부모, 형제자매 | 3일 |
② 경조휴가는 경조사 발생일로부터 30일 이내에 사용하여야 한다.
③ 경조휴가는 인사시스템에서 신청하며, 증빙서류를 첨부하여야 한다.
제27조(출산전후휴가) ① 임신 중인 직원에게는 출산 전과 출산 후를 통하여 90일(한 번에 둘 이상 자녀를 임신한 경우에는 120일)의 출산전후휴가를 준다.
② 제1항의 휴가 기간 중 최초 60일은 유급으로 한다.
제28조(육아휴직) ① 회사는 만 8세 이하 또는 초등학교 2학년 이하의 자녀를 양육하기 위하여 신청하는 직원에게 1년 이내의 육아휴직을 허용한다.
② 육아휴직을 신청하려는 직원은 휴직 개시 예정일 30일 전까지 인사팀에 신청서를 제출하여야 한다.
부칙
제1조(시행일) 이 규칙은 2026년 1월 1일부터 시행한다.
""",
"SAMPLE_누리정밀_경조금지급규정.txt": """누리정밀 경조금 지급 규정
제1조(목적) 이 규정은 직원의 경조사에 대한 경조금 지급 기준과 절차를 정함을 목적으로 한다.
제3조(지급대상) 경조금은 재직 중인 직원에게 지급한다. 휴직 중인 직원도 지급대상에 포함한다.
제4조(지급기준) 경조금은 다음 표에 따라 지급한다.
| 구분 | 대상 | 경조금 | 화환 |
|---|---|---|---|
| 결혼 | 본인 | 500,000원 | 지원 |
| 결혼 | 자녀 | 300,000원 | 지원 |
| 출산 | 본인 또는 배우자 | 300,000원 | - |
| 회갑·칠순·팔순 | 본인 및 배우자의 부모 | 200,000원 | - |
| 사망 | 배우자, 본인 및 배우자의 부모 | 1,000,000원 | 조화 지원 |
제5조(신청) ① 경조금을 받으려는 직원은 경조사 발생일로부터 60일 이내에 인사시스템으로 신청하여야 한다.
② 신청 시 청첩장, 출생증명서, 가족관계증명서 등 증빙서류를 첨부하여야 한다.
제6조(중복지급의 제한) 부부가 모두 직원인 경우 같은 경조사에 대한 경조금은 1인에게만 지급한다.
""",
"SAMPLE_누리정밀_여비규정.txt": """누리정밀 여비 규정
제2조(정의) 이 규정에서 "일비"란 출장 중 현지 교통비, 통신비 등 부대비용을 충당하기 위하여 정액으로 지급하는 금액을 말한다.
제8조(국내출장 여비) 국내출장 여비는 다음 표에 따라 지급한다.
| 직급 | 일비(1일) | 식비(1일) | 숙박비 상한(1박) |
|---|---|---|---|
| 임원 | 40,000원 | 40,000원 | 실비 |
| 부장 이하 | 25,000원 | 25,000원 | 서울 100,000원 / 그 외 80,000원 |
제9조(해외출장 여비) ① 해외출장 숙박비는 별표 2의 지역별 상한액 범위에서 실비로 지급한다.
② 해외출장 일비는 미화 50달러로 한다.
제12조(여비의 정산) ① 출장자는 출장을 마친 날부터 7일 이내에 영수증을 첨부하여 여비를 정산하여야 한다.
② 정산하지 아니한 가지급금은 다음 달 급여에서 공제한다.
제13조(휴일 출장) 휴일에 출장한 경우에는 근무한 것으로 보며, 출장 복귀 후 1개월 이내에 대체휴일을 부여한다.
""",
"SAMPLE_누리정밀_교육훈련규정.txt": """누리정밀 교육훈련 규정
제3조(신입사원 교육) 신입사원은 입사 후 3개월 이내에 직무기초교육과 법정의무교육(성희롱 예방, 개인정보보호, 산업안전)을 이수하여야 한다.
제10조(외부교육비 지원) ① 회사는 직무와 관련된 외부교육을 수강하는 직원에게 1인당 연 2,000,000원 한도에서 교육비를 지원한다.
② 대학원 학비는 근속 3년 이상인 직원에게 학기당 등록금의 50퍼센트를 지원한다.
제11조(교육비의 반납) 대학원 학비를 지원받은 직원이 졸업 후 2년 이내에 퇴직하는 경우 지원받은 금액을 반납하여야 한다.
""",
}

EVAL_COLS = ["qid", "question", "intent", "domain", "event_type", "target_doc", "target_article",
             "context_type", "needs_tool", "history", "expected_route"]
_D = "SAMPLE_누리정밀_"
SAMPLE_EVAL = [
 ("S01","입사한 지 1년 안 됐는데 연차 쓸 수 있어요?","규정","근태관리","입사·승진",_D+"취업규칙","제25조","조문",False,"",""),
 ("S02","반차는 몇 시부터 몇 시까지예요?","규정","근태관리","없음",_D+"취업규칙","제25조","조문",False,"",""),
 ("S03","못 쓴 연차는 돈으로 받을 수 있나요?","규정","급여보상","없음",_D+"취업규칙","제25조의2","조문",False,"",""),
 ("S04","결혼하면 며칠 쉴 수 있어요?","규정","근태관리","결혼",_D+"취업규칙","제26조","표",False,"",""),
 ("S05","다음 달 결혼하는데 회사에서 받을 수 있는 거 다 알려줘","이벤트","복리후생","결혼",f"{_D}취업규칙;{_D}경조금지급규정","제26조;제4조","표",False,"",""),
 ("S06","아내가 애기 낳으면 저는 휴가 며칠이에요?","규정","근태관리","출산",_D+"취업규칙","제26조","표",False,"",""),
 ("S07","아버지 팔순인데 축하금 나와요?","규정","복리후생","없음",_D+"경조금지급규정","제4조","표",False,"",""),
 ("S08","경조금 신청은 언제까지 해야 돼요?","규정","복리후생","없음",_D+"경조금지급규정","제5조","조문",False,"",""),
 ("S09","국내 출장 가면 일비 얼마 나와요?","규정","업무지원","없음",_D+"여비규정","제8조","표",False,"",""),
 ("S10","출장비 정산 기한이 어떻게 돼요?","규정","업무지원","없음",_D+"여비규정","제12조","조문",False,"",""),
 ("S11","주말에 출장 가면 근무로 쳐줘요?","규정","업무지원","없음",_D+"여비규정","제13조","조문",False,"",""),
 ("S12","대학원 학비 지원되나요?","규정","교육·평가","없음",_D+"교육훈련규정","제10조","조문",False,"",""),
 ("S13","육아휴직은 언제까지 신청해야 해요?","규정","근태관리","휴직·퇴직",_D+"취업규칙","제28조","조문",False,"",""),
 ("S14","반려동물 수당 있어요?","규정","급여보상","없음","","","문서에없음",False,"",""),
 ("S15","제 연차 몇 개 남았어요?","내정보","근태관리","없음","","","DB",True,"",""),
 # ── v2에서 추가: 줄임말 · 멀티턴 · 복합 · 둘다 · 범위밖 · 민감 · 되묻기
 ("S16","육휴 신청은 며칠 전까지 해야 돼?","규정","근태관리","휴직·퇴직",_D+"취업규칙","제28조","조문",False,"",""),
 ("S17","그럼 저는 휴가 며칠 받아요?","규정","근태관리","출산",_D+"취업규칙","제26조","표",False,"와이프가 다음 달에 출산 예정이에요",""),
 ("S18","결혼하면 휴가 며칠이고 경조금은 얼마예요?","이벤트","복리후생","결혼",f"{_D}취업규칙;{_D}경조금지급규정","제26조;제4조","표",False,"",""),
 ("S19","제 연차 몇 개 남았는지랑 못 쓰면 수당 나오는지 알려줘","둘다","급여보상","없음",_D+"취업규칙","제25조의2","조문",True,"",""),
 ("S20","오늘 점심 메뉴 추천해줘","범위밖","","없음","","","범위밖",False,"",""),
 ("S21","팀장님한테 계속 폭언 들어요 어디에 말해야 돼요?","민감","","없음","","","담당자",False,"",""),
 ("S22","경조금 얼마 나와요?","규정","복리후생","없음",_D+"경조금지급규정","제4조","표",False,"","되묻기"),
 ("S23","부부가 둘 다 우리 회사 다니면 경조금 둘 다 받아요?","규정","복리후생","없음",_D+"경조금지급규정","제6조","조문",False,"",""),
 ("S24","해외 출장 일비는 얼마예요?","규정","업무지원","없음",_D+"여비규정","제9조","조문",False,"",""),
 ("S25","구내식당 밥값 지원돼요?","규정","복리후생","없음","","","문서에없음",False,"",""),
 ("S26","지원받고 바로 퇴사하면 어떻게 돼요?","규정","교육·평가","휴직·퇴직",_D+"교육훈련규정","제11조","조문",False,"대학원 다니려고 하는데요",""),
 ("S27","주식 투자 어떻게 시작해요?","범위밖","","없음","","","범위밖",False,"",""),
]

data_dir = Path(CONFIG["DATA_DIR"])
REG_EXTS = {".txt", ".md", ".pdf", ".hwpx", ".hwp"}
real_files = [p for p in data_dir.rglob("*") if p.suffix.lower() in REG_EXTS and not p.name.startswith("SAMPLE_")]
USING_SAMPLE = len(real_files) == 0
if USING_SAMPLE:
    for name, text in SAMPLE_DOCS.items():
        (data_dir / name).write_text(text, encoding="utf-8")
    CONFIG["EVAL_PATH"] = "data/SAMPLE_eval_set.csv"
    pd.DataFrame(SAMPLE_EVAL, columns=EVAL_COLS).assign(target_answer="").to_csv(
        CONFIG["EVAL_PATH"], index=False, encoding="utf-8-sig")
    print("⚠️ 실제 규정 파일이 없어 가상 샘플로 실행합니다. 이 점수는 보고용이 아닙니다.")
else:
    for p in data_dir.glob("SAMPLE_*"):
        p.unlink()
    print(f"실제 규정 파일 {len(real_files)}개를 사용합니다.")

⚠️ 실제 규정 파일이 없어 가상 샘플로 실행합니다. 이 점수는 보고용이 아닙니다.


### 문서 로딩 · 조항 단위 청킹 (A0와 동일)
A0 노트북과 **같은 코드**입니다. 청킹이 바뀌면 A0 점수와 비교할 수 없으니, 바꿀 때는 두 노트북을 같이 바꾸세요.

In [32]:
def table_to_md(rows):
    rows = [[(c or "").replace("\n", " ").strip() for c in r] for r in rows if r and any((c or "").strip() for c in r)]
    if not rows:
        return ""
    w = max(len(r) for r in rows)
    rows = [r + [""] * (w - len(r)) for r in rows]
    lines = ["| " + " | ".join(rows[0]) + " |", "|" + "---|" * w]
    lines += ["| " + " | ".join(r) + " |" for r in rows[1:]]
    return "\n".join(lines)

def load_pdf(path):
    import pdfplumber
    out = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            tables = page.find_tables()
            bboxes = [t.bbox for t in tables]
            def outside(obj):
                cx, cy = (obj["x0"] + obj["x1"]) / 2, (obj["top"] + obj["bottom"]) / 2
                return not any(x0 <= cx <= x1 and t <= cy <= b for x0, t, x1, b in bboxes)
            blocks = [(ln["top"], ln["text"]) for ln in page.filter(outside).extract_text_lines()]
            blocks += [(t.bbox[1], table_to_md(t.extract())) for t in tables]
            out += [txt for _, txt in sorted(blocks, key=lambda b: b[0]) if txt.strip()]
    return "\n".join(out)

def _local(tag):
    return tag.rsplit("}", 1)[-1]

def load_hwpx(path):
    out = []
    with zipfile.ZipFile(path) as z:
        sections = sorted(n for n in z.namelist() if re.match(r"Contents/section\d+\.xml", n))
        for name in sections:
            root = ET.fromstring(z.read(name))
            for p in root:                      # 최상위 문단 단위
                if _local(p.tag) != "p":
                    continue
                tbls = [e for e in p.iter() if _local(e.tag) == "tbl"]
                if tbls:
                    for tbl in tbls:
                        rows = []
                        for tr in (e for e in tbl.iter() if _local(e.tag) == "tr"):
                            rows.append(["".join(t.text or "" for t in tc.iter() if _local(t.tag) == "t")
                                         for tc in (e for e in tr if _local(e.tag) == "tc")])
                        out.append(table_to_md(rows))
                else:
                    txt = "".join(t.text or "" for t in p.iter() if _local(t.tag) == "t").strip()
                    if txt:
                        out.append(txt)
    return "\n".join(out)

def load_hwp(path):
    if shutil.which("hwp5txt"):
        return subprocess.run(["hwp5txt", str(path)], capture_output=True, text=True, check=True).stdout
    raise RuntimeError(f"{path.name}: .hwp는 hwp5txt(pip install pyhwp)가 필요합니다. "
                       "한글에서 HWPX나 PDF로 저장해 넣는 것을 권장합니다.")

LOADERS = {".txt": lambda p: p.read_text(encoding="utf-8"), ".md": lambda p: p.read_text(encoding="utf-8"),
           ".pdf": load_pdf, ".hwpx": load_hwpx, ".hwp": load_hwp}

def normalize_text(t):
    t = t.replace(" ", " ").replace("　", " ")
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"^\s*-\s*\d+\s*-\s*$", "", t, flags=re.M)   # 쪽번호 "- 3 -"
    return re.sub(r"\n{2,}", "\n", t).strip()

docs, load_errors = [], []
for p in sorted(Path(CONFIG["DATA_DIR"]).rglob("*")):
    if p.suffix.lower() not in LOADERS:
        continue
    try:
        docs.append({"doc_title": p.stem, "path": str(p), "text": normalize_text(LOADERS[p.suffix.lower()](p))})
    except Exception as e:
        load_errors.append((p.name, str(e)))

print(f"로딩 성공 {len(docs)}개 / 실패 {len(load_errors)}개")
for name, err in load_errors:
    print("  ✗", name, "→", err)
pd.DataFrame([{"doc_title": d["doc_title"], "chars": len(d["text"])} for d in docs])

로딩 성공 4개 / 실패 0개


,doc_title,chars
0,SAMPLE_누리정밀_경조금지급규정,536
1,SAMPLE_누리정밀_교육훈련규정,287
2,SAMPLE_누리정밀_여비규정,505
3,SAMPLE_누리정밀_취업규칙,1213


In [33]:
ART_RE = re.compile(r"^\s*(제\s*(\d+)\s*조(?:\s*의\s*(\d+))?)\s*(?:\(([^)]{1,40})\)|(?![에의및을를은는이가과와도로부까]))")
CHAP_RE = re.compile(r"^\s*(제\s*\d+\s*장)\s*(.*)$")
SUPP_RE = re.compile(r"^\s*부\s*칙\s*(?:[<(\[（].*)?$")
CIRCLED = "①②③④⑤⑥⑦⑧⑨⑩⑪⑫⑬⑭⑮⑯⑰⑱⑲⑳"

def norm_article(s):
    s = re.sub(r"\s+", "", str(s))
    return s.replace("부칙", "부칙 ") if s.startswith("부칙") else s

def split_blocks(lines):
    """줄 목록 → 블록 목록. 표는 한 블록, 항(①…)이 시작될 때마다 새 블록."""
    blocks, cur, in_table = [], [], False
    for ln in lines:
        is_tbl = ln.lstrip().startswith("|")
        new_para = ln.lstrip()[:1] in CIRCLED
        if cur and (is_tbl != in_table or (new_para and not is_tbl)):
            blocks.append((in_table, cur)); cur = []
        cur.append(ln); in_table = is_tbl
    if cur:
        blocks.append((in_table, cur))
    return blocks

def split_table(lines, max_chars):
    head, body = lines[:2], lines[2:]
    parts, cur = [], []
    for row in body:
        if cur and len("\n".join(head + cur + [row])) > max_chars:
            parts.append(head + cur); cur = []
        cur.append(row)
    return parts + [head + cur] if cur else parts

def pack(blocks, max_chars):
    """블록을 순서대로 max_chars 안에서 묶음. 긴 표는 헤더 반복 분할."""
    units = []
    for is_tbl, ls in blocks:
        if is_tbl and len("\n".join(ls)) > max_chars and len(ls) > 3:
            units += [(True, p) for p in split_table(ls, max_chars)]
        else:
            units.append((is_tbl, ls))
    parts, cur, cur_tbl = [], [], False
    for is_tbl, ls in units:
        if cur and len("\n".join(cur + ls)) > max_chars:
            parts.append((cur_tbl, cur)); cur, cur_tbl = [], False
        cur += ls; cur_tbl = cur_tbl or is_tbl
    if cur:
        parts.append((cur_tbl, cur))
    return parts

def chunk_document(doc, max_chars):
    title = doc["doc_title"]
    sections, chapter, supp = [], "", False
    cur = {"article": "서문", "article_title": "", "chapter": "", "lines": []}
    last_num = -1
    for ln in doc["text"].split("\n"):
        m_ch, m_sp, m_ar = CHAP_RE.match(ln), SUPP_RE.match(ln), ART_RE.match(ln)
        if m_ch and not m_ar:
            chapter = f"{re.sub(r'\s+', '', m_ch.group(1))} {m_ch.group(2).strip()}".strip()
            continue
        if m_sp:
            supp, last_num = True, -1
            continue
        if m_ar:
            num = int(m_ar.group(2)) + (int(m_ar.group(3)) / 100 if m_ar.group(3) else 0)
            if num > last_num:                                 # 번호가 거꾸로 가면 인용으로 간주
                if cur["lines"]:
                    sections.append(cur)
                art = norm_article(m_ar.group(1))
                cur = {"article": ("부칙 " if supp else "") + art,
                       "article_title": (m_ar.group(4) or "").strip(),
                       "chapter": "부칙" if supp else chapter, "lines": [ln.strip()]}
                last_num = num
                continue
        cur["lines"].append(ln.strip())
    if cur["lines"]:
        sections.append(cur)

    chunks = []
    for sec in sections:
        header = f"[{title}] {sec['article']}" + (f"({sec['article_title']})" if sec["article_title"] else "")
        parts = pack(split_blocks(sec["lines"]), max_chars)
        for i, (has_tbl, ls) in enumerate(parts):
            body = "\n".join(ls)
            if sec["article"] == "서문" and len(body) < 30:   # 문서 제목 한 줄짜리 서문은 버림
                continue
            chunks.append({
                "chunk_id": f"{title}::{sec['article']}::{i}",
                "doc_title": title, "article": sec["article"], "article_title": sec["article_title"],
                "chapter": sec["chapter"], "part": i, "n_parts": len(parts), "has_table": has_tbl,
                "text": body, "embed_text": f"{header}\n{body}" if i == 0 else f"{header} (이어짐)\n{body}",
            })
    return chunks

chunks = [c for d in docs for c in chunk_document(d, CONFIG["CHUNK_MAX_CHARS"])]
chunk_df = pd.DataFrame(chunks)
chunk_df["n_chars"] = chunk_df["text"].str.len()
print(f"청크 {len(chunk_df)}개 | 문서 {chunk_df.doc_title.nunique()}개 | 표 포함 {int(chunk_df.has_table.sum())}개")
chunk_df.groupby("doc_title").agg(chunks=("chunk_id", "count"), articles=("article", "nunique"),
                                  mean_chars=("n_chars", "mean"), max_chars=("n_chars", "max")).round(0)

청크 21개 | 문서 4개 | 표 포함 3개


,chunks,articles,mean_chars,max_chars
doc_title,,,,
SAMPLE_누리정밀_경조금지급규정,5,5,103.0,251
SAMPLE_누리정밀_교육훈련규정,3,3,91.0,124
SAMPLE_누리정밀_여비규정,5,5,98.0,182
SAMPLE_누리정밀_취업규칙,8,8,147.0,322


In [34]:
print("가장 긴 청크 5개:")
display(chunk_df.nlargest(5, "n_chars")[["chunk_id", "n_chars", "has_table"]])
print("서문(조항 헤더 앞) 청크 — 많으면 헤더 인식 실패 의심:")
display(chunk_df[chunk_df.article == "서문"][["chunk_id", "n_chars"]])
dup = chunk_df[chunk_df.part == 0].duplicated(["doc_title", "article"], keep=False)
if dup.any():
    print("⚠️ 같은 문서 안에 같은 조항 번호가 두 번 나옵니다 (별표·서식 혼입 가능):")
    display(chunk_df[chunk_df.part == 0][dup][["chunk_id"]])
print(chunk_df.iloc[min(3, len(chunk_df) - 1)]["embed_text"])

chunk_df.assign(preview=chunk_df.text.str.slice(0, 80))[
    ["chunk_id", "doc_title", "article", "article_title", "chapter", "part", "has_table", "n_chars", "preview"]
].to_csv(Path(CONFIG["RESULTS_DIR"]) / "chunks.csv", index=False, encoding="utf-8-sig")

가장 긴 청크 5개:


,chunk_id,n_chars,has_table
15,SAMPLE_누리정밀_취업규칙::제25조::0,322,False
17,SAMPLE_누리정밀_취업규칙::제26조::0,280,True
2,SAMPLE_누리정밀_경조금지급규정::제4조::0,251,True
9,SAMPLE_누리정밀_여비규정::제8조::0,182,True
16,SAMPLE_누리정밀_취업규칙::제25조의2::0,138,False


서문(조항 헤더 앞) 청크 — 많으면 헤더 인식 실패 의심:


,chunk_id,n_chars


[SAMPLE_누리정밀_경조금지급규정] 제5조(신청)
제5조(신청) ① 경조금을 받으려는 직원은 경조사 발생일로부터 60일 이내에 인사시스템으로 신청하여야 한다.
② 신청 시 청첩장, 출생증명서, 가족관계증명서 등 증빙서류를 첨부하여야 한다.


## 2. 사람이 관리하는 사전
시스템은 이 사전과 BGE-M3만 보고 자동으로 재작성·분류합니다. 사람은 **처음에 한 번 만들고, 평가에서 놓친 표현이 나오면 조금씩 추가**하면 됩니다.

| 사전 | 쓰는 곳 | 예 |
|---|---|---|
| `ABBR_DICT` | 재작성: 줄임말을 규정 용어로 **바꿈** | 육휴 → 육아휴직 |
| `TERM_DICT` | 재작성: 구어체가 보이면 규정 용어를 **뒤에 덧붙임** (원문은 그대로 둠) | "쉴 수" → +휴가 |
| `DOMAIN_KEYWORDS` | 분류(업무 영역) · 청크 태깅 | 출장, 일비 → 업무지원 |
| `EVENT_DICT` | 분류(생애 이벤트) · 청크 태깅 · 멀티턴 이어받기 | 와이프 출산 → 출산 |
| `SLOT_DICT` | 조건(대상·지역·직급) 추출 → 재작성에 덧붙임, LLM에 조건 전달 | 와이프 → 배우자 |
| 개인 조회 · 민감 · 되묻기 규칙 | 라우터 | "제 연차 몇 개 남았어요?" → 개인조회 |
| `PROTO_QUESTIONS` | 분류: 영역별 **대표 질문** → BGE-M3로 평균 벡터(프로토타입) | 영역마다 8~20문장 |

⚠️ **주의**
- 업무 영역 키는 A0 샘플에 나온 5개만 넣었습니다. **노션의 6개 업무 영역 이름에 맞춰** 키를 고치고 빠진 영역을 추가하세요. 평가셋 `domain` 값과 글자까지 같아야 분류 정확도가 계산됩니다.
- 사전과 대표 질문은 **규정 문서를 보고** 만드세요. 평가셋 질문을 보고 만들면 점수가 부풀려집니다. 아래 값은 샘플 규정용 예시입니다.

In [35]:
ABBR_DICT = {   # 줄임말 → 규정 용어 (치환)
    "육휴": "육아휴직", "출휴": "출산전후휴가", "출산휴가": "출산전후휴가",
    "경조비": "경조금", "연차수당": "연차휴가 미사용 수당", "출장비": "출장 여비",
}
TERM_DICT = {   # 구어체 표현 → 덧붙일 규정 용어
    "쉴 수": "휴가", "쉬는": "휴가", "쉬어": "휴가",
    "애기 낳": "출산", "아기 낳": "출산", "애 낳": "출산",
    "축하금": "경조금", "부조": "경조금", "조의금": "경조금",
    "돈으로": "수당 보상", "못 쓴 연차": "연차휴가 미사용 수당", "못 쓰면": "미사용 수당",
    "주말": "휴일", "근무로 쳐": "근무한 것으로 본다",
    "학비": "교육비 지원", "등록금": "학비", "숙소": "숙박비", "밥값": "식비",
    "퇴사": "퇴직", "그만두": "퇴직",
    "팔순": "회갑·칠순·팔순", "칠순": "회갑·칠순·팔순", "환갑": "회갑",
    "받을 수 있는 거": "지원 휴가 경조금",
}
DOMAIN_KEYWORDS = {   # 노션 6개 업무 영역에 맞춰 수정하세요
    "근태관리": ["연차", "반차", "휴가", "휴직", "육아휴직", "출산전후휴가", "경조휴가", "근무시간",
                 "출근", "퇴근", "지각", "대체휴일"],
    "급여보상": ["급여", "월급", "임금", "수당", "통상임금", "상여", "성과급", "연봉"],
    "복리후생": ["경조금", "경조사", "화환", "조화", "복지포인트", "건강검진", "동호회", "학자금"],
    "업무지원": ["출장", "여비", "일비", "숙박비", "식비", "정산", "가지급금", "법인카드"],
    "교육·평가": ["교육", "교육비", "연수", "대학원", "학비", "자격증", "이수", "인사평가", "승진시험"],
}
EVENT_DICT = {        # 평가셋 event_type 값과 같게 유지
    "결혼": ["결혼", "혼인", "청첩장", "신혼"],
    "출산": ["출산", "임신", "애기", "아기", "출생", "낳"],
    "휴직·퇴직": ["휴직", "퇴직", "퇴사", "그만두"],
    "입사·승진": ["입사", "신입", "승진", "1년 미만", "수습"],
}
SLOT_DICT = {         # 조건 → 표준값
    "대상": {"본인": ["본인"], "배우자": ["배우자", "아내", "와이프", "남편", "부인"],
             "자녀": ["자녀", "아들", "딸"], "부모": ["부모", "아버지", "어머니", "아빠", "엄마", "장인", "장모"],
             "조부모": ["할아버지", "할머니", "조부모"]},
    "지역": {"해외": ["해외", "외국", "국외"], "국내": ["국내", "지방"]},
    "직급": {"임원": ["임원", "상무", "전무"], "부장 이하": ["부장", "차장", "과장", "대리", "사원"]},
}

# 개인 조회: 1인칭 + 조회 표현이 같이 있으면 개인 기록 질문
FIRST_PERSON_RE = re.compile(r"(?:^|\s)(?:내|나|제|저|나의|저의|내가|제가|저는|나는)(?=\s|$)")
LOOKUP_WORDS = ["남았", "남은", "잔여", "내역", "현황", "승인", "지급됐", "들어왔", "처리됐", "얼마나 썼", "몇 개 썼", "조회"]
REG_CUES = ["수당", "규정", "기준", "받을 수", "되나요", "돼요", "가능", "나와", "나오", "언제까지", "얼마"]   # 개인 조회 + 이게 있으면 "둘다"
SENSITIVE_WORDS = ["폭언", "괴롭힘", "성추행", "폭행", "해고", "부당", "고소", "소송", "차별"]   # → 담당자 연결

# 조건이 빠지면 답이 갈리는 질문 → 템플릿으로 되묻기
CLARIFY_RULES = [
    {"name": "경조사 종류",
     "if_any": ["경조금", "경조휴가", "경조사"],
     "unless_any": sum(EVENT_DICT.values(), []) + ["사망", "장례", "회갑", "칠순", "팔순", "조의", "부고",
                   "신청", "언제까지", "서류", "증빙", "기한", "중복", "부부", "둘 다", "대상"],
     "ask": "어떤 경조사인지 알려주시겠어요? (결혼 / 출산 / 회갑·칠순·팔순 / 사망) 대상(본인·자녀·부모 등)도 같이 알려주시면 정확히 안내해 드릴게요."},
]
ANAPHORA = ["그거", "그건", "그것", "그럼", "그러면", "거기", "그때", "그 경우", "이건", "이거"]

PROTO_QUESTIONS = {   # 영역별 대표 질문. 평가셋 질문을 그대로 넣지 마세요.
    "근태관리": ["연차는 1년에 몇 개 나오나요?", "반차 쓰려면 어떻게 해야 해요?", "휴가 신청은 어디서 하나요?",
                 "육아휴직 기간은 최대 얼마나 돼요?", "출근 시간이 몇 시예요?", "지각하면 어떻게 처리돼요?",
                 "경조휴가 며칠 쓸 수 있어요?", "휴직하고 복직하는 절차가 궁금해요", "출산휴가는 며칠이에요?",
                 "대체휴일은 언제 쓸 수 있나요?"],
    "급여보상": ["월급날이 언제예요?", "야근하면 수당 나오나요?", "통상임금에 뭐가 포함돼요?",
                 "연차수당은 어떻게 계산해요?", "성과급 지급 기준이 뭐예요?", "휴일에 일하면 수당 받나요?",
                 "급여명세서는 어디서 봐요?", "상여금은 언제 나와요?"],
    "복리후생": ["결혼하면 회사에서 축하금 주나요?", "부모님 돌아가시면 조의금 얼마예요?",
                 "복지포인트는 어디에 쓸 수 있어요?", "건강검진 지원돼요?", "경조금 신청 서류가 뭐예요?",
                 "동호회 지원금 있나요?", "자녀 학자금 지원되나요?", "화환 보내주나요?"],
    "업무지원": ["출장 가면 식비 얼마 나와요?", "출장비 정산은 어떻게 해요?", "해외출장 숙박비 한도가 얼마예요?",
                 "법인카드 써도 되나요?", "출장 일비가 뭐예요?", "가지급금 받을 수 있나요?",
                 "출장 중 교통비는 어떻게 처리해요?"],
    "교육·평가": ["외부 교육 들으면 비용 지원돼요?", "신입사원 교육은 뭐 들어야 해요?", "자격증 따면 지원금 있어요?",
                 "법정의무교육은 뭐가 있나요?", "대학원 등록금 지원 조건이 뭐예요?", "인사평가는 언제 해요?",
                 "온라인 강의도 교육비 지원되나요?"],
    "범위밖": ["오늘 날씨 어때?", "점심 뭐 먹을까?", "주식 추천해줘", "재미있는 영화 알려줘", "회사 근처 맛집 어디야?",
               "너는 누구야?", "파이썬 코드 짜줘", "주말에 뭐 하지?", "커피 한잔 할래?", "로또 번호 알려줘"],
}
MSG = {
    "범위밖": "저는 사내 복리후생·인사 규정 안내를 도와드리고 있어요. 휴가, 수당, 경조사, 출장, 교육 같은 제도에 대해 물어봐 주세요.",
    "개인조회": "개인 기록(잔여 연차, 지급 내역, 신청 현황 등)은 인사시스템의 [내 정보]에서 확인하실 수 있어요.",
    "담당자연결": "이 문의는 인사팀 담당자가 직접 도와드리는 게 좋겠어요. 인사팀(내선 0000)으로 연락해 주세요. 상담 내용은 비공개로 처리됩니다.",
    "답변불가": "규정에서 관련 내용을 찾지 못했어요. 인사팀에 문의해 주세요.",
}
assert set(PROTO_QUESTIONS) - {"범위밖"} == set(DOMAIN_KEYWORDS), "PROTO_QUESTIONS와 DOMAIN_KEYWORDS의 영역 이름을 맞춰 주세요"

## 3. BGE-M3 인덱스
BGE-M3는 **한 번 인코딩으로 세 가지 표현**을 줍니다. 그래서 별도 리랭커 모델 없이 BGE-M3 하나로 검색과 재정렬을 모두 합니다.
- **dense**: 문장 벡터 1개 → 의미 검색
- **sparse (lexical weights)**: 토큰별 가중치 → 키워드 검색 (BM25 역할)
- **ColBERT**: 토큰마다 벡터 → 질문 토큰마다 가장 비슷한 문서 토큰을 찾아 평균 → 꼼꼼한 재정렬

청크 인코딩 결과는 `cache/`에 저장해서, 청크·모델이 같으면 다시 계산하지 않습니다. ColBERT 벡터는 메모리를 아끼려고 float16으로 저장합니다.
`EMBEDDER="mock"`은 모델 없이 코드만 점검하는 가짜 인코더라 점수에 의미가 없습니다.

In [36]:
class M3Encoder:
    """BGE-M3 한 번 호출로 dense / sparse / ColBERT를 모두 뽑습니다."""
    def __init__(self, cfg):
        import torch
        from FlagEmbedding import BGEM3FlagModel
        self.device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
        # CPU 환경일 경우 fp16을 비활성화하여 안정적으로 기동합니다.
        use_fp16 = (self.device == "cuda")
        self.model = BGEM3FlagModel(cfg["MODEL_NAME"], use_fp16=use_fp16, devices=self.device)
        self.name, self.max_len, self.bs = cfg["MODEL_NAME"], cfg["MAX_SEQ_LEN"], cfg["BATCH_SIZE"]

    def encode(self, texts, colbert=True):
        out = self.model.encode(list(texts), batch_size=self.bs, max_length=self.max_len,
                                return_dense=True, return_sparse=True, return_colbert_vecs=colbert)
        dense = np.asarray(out["dense_vecs"], dtype=np.float32).reshape(len(texts), -1)
        dense /= np.maximum(np.linalg.norm(dense, axis=1, keepdims=True), 1e-12)
        sparse = [{str(k): float(v) for k, v in d.items()} for d in out["lexical_weights"]]
        cols = None
        if colbert:
            cols = []
            for c in out["colbert_vecs"]:
                c = np.asarray(c, dtype=np.float32)
                cols.append((c / np.maximum(np.linalg.norm(c, axis=1, keepdims=True), 1e-12)).astype(np.float16))
        return dense, sparse, cols


class MockM3:
    """모델 없이 파이프라인 흐름만 점검하는 가짜 BGE-M3 (문자 n-gram). 점수는 의미 없음."""
    def __init__(self, cfg):
        from sklearn.feature_extraction.text import HashingVectorizer
        self.hv = HashingVectorizer(analyzer="char_wb", ngram_range=(2, 3), n_features=2**14,
                                    alternate_sign=False, norm=None)
        self.name, self.device, self.idf = "mock-char-ngram", "cpu", None

    def fit(self, texts):
        X = (self.hv.transform(texts) > 0).astype(np.float32)
        df = np.asarray(X.sum(0)).ravel()
        self.idf = (np.log((1 + X.shape[0]) / (1 + df)) + 1).astype(np.float32)
        return self

    @staticmethod
    def _bigrams(t):
        t = re.sub(r"[^\w]", "", t)
        return [t[i:i + 2] for i in range(len(t) - 1)] or [t]

    @staticmethod
    def _tokvec(tok):
        rng = np.random.default_rng(int(hashlib.md5(tok.encode()).hexdigest()[:8], 16))
        v = rng.standard_normal(64).astype(np.float32)
        return v / np.linalg.norm(v)

    def encode(self, texts, colbert=True):
        texts = list(texts)
        dense = self.hv.transform(texts).multiply(self.idf).toarray().astype(np.float32)
        dense /= np.maximum(np.linalg.norm(dense, axis=1, keepdims=True), 1e-12)
        sparse = []
        for t in texts:
            bg = set(self._bigrams(t))
            sparse.append({b: 1 / np.sqrt(len(bg)) for b in bg})
        cols = [np.stack([self._tokvec(b) for b in self._bigrams(t)[:256]]).astype(np.float16) for t in texts] if colbert else None
        return dense, sparse, cols


proto_texts = [q for qs in PROTO_QUESTIONS.values() for q in qs]
if CONFIG["EMBEDDER"] == "bge-m3":
    enc = M3Encoder(CONFIG)
else:
    enc = MockM3(CONFIG).fit(chunk_df.embed_text.tolist() + proto_texts)
    print("⚠️ MOCK 모드 — 코드 점검용입니다. 베이스라인 점수로 쓰지 마세요.")

fingerprint = hashlib.md5((enc.name + str(CONFIG["MAX_SEQ_LEN"]) + "".join(chunk_df.chunk_id)
                           + "".join(chunk_df.embed_text)).encode()).hexdigest()[:12]
cache_path = Path(CONFIG["CACHE_DIR"]) / f"m3_index_{fingerprint}.pkl"
t0 = time.perf_counter()
if cache_path.exists() and CONFIG["EMBEDDER"] == "bge-m3":
    D_DENSE, D_SPARSE, D_COLBERT = pickle.loads(cache_path.read_bytes())
    print(f"캐시에서 불러옴: {cache_path}")
else:
    D_DENSE, D_SPARSE, D_COLBERT = enc.encode(chunk_df.embed_text.tolist(), colbert=True)
    if CONFIG["EMBEDDER"] == "bge-m3":
        cache_path.write_bytes(pickle.dumps((D_DENSE, D_SPARSE, D_COLBERT)))
index_sec = time.perf_counter() - t0
N = len(chunk_df)
colbert_mb = sum(c.nbytes for c in D_COLBERT) / 2**20
print(f"청크 {N}개 인코딩 {index_sec:.1f}초 | dense {D_DENSE.shape} | ColBERT {colbert_mb:.0f}MB | device={enc.device}")

# sparse 역색인 (질문 토큰 → 청크)
SPARSE_INV = defaultdict(list)
for i, d in enumerate(D_SPARSE):
    for t, w in d.items():
        SPARSE_INV[t].append((i, w))

# 질문 인코딩 캐시 (같은 질문을 여러 구성에서 평가하므로)
_QCACHE = {}
def encode_query(text):
    if text not in _QCACHE:
        t0 = time.perf_counter()
        d, s, c = enc.encode([text], colbert=True)
        _QCACHE[text] = {"dense": d[0], "sparse": s[0], "colbert": c[0].astype(np.float32),
                         "ms": (time.perf_counter() - t0) * 1000}
    return _QCACHE[text]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

캐시에서 불러옴: cache/m3_index_2e8f1ba33bc6.pkl
청크 21개 인코딩 0.0초 | dense (21, 1024) | ColBERT 4MB | device=cpu


### 프로토타입 · 청크 태깅
- **프로토타입**: 영역별 대표 질문의 dense 벡터 평균. 질문과의 코사인 유사도로 영역을 고릅니다.
- **청크 태깅**: 필터가 작동하려면 청크에도 같은 라벨이 있어야 합니다. 조항 제목·본문에 영역 키워드가 있으면 그 영역(여러 개 가능), 없으면 가장 가까운 프로토타입 영역을 붙입니다. 이벤트도 키워드로 붙입니다.
- 결과는 `results/chunk_tags.csv`에 저장됩니다. 틀린 태그는 `data/chunk_tags_override.csv`(컬럼: `chunk_id, domains, events`, 여러 개는 `;`)로 고치면 다음 실행부터 반영됩니다.

In [28]:
def kw_hits(text, kw_dict):
    """사전 키 → 텍스트에 나온 키워드 수 (0이면 제외)"""
    out = {}
    for k, words in kw_dict.items():
        n = sum(text.count(w) for w in words)
        if n:
            out[k] = n
    return out

PROTO_LABELS = list(PROTO_QUESTIONS)
_pd, _, _ = enc.encode(proto_texts, colbert=False)
PROTO_VECS = _pd
_ofs, P = 0, []
for lab in PROTO_LABELS:
    n = len(PROTO_QUESTIONS[lab])
    v = _pd[_ofs:_ofs + n].mean(0); _ofs += n
    P.append(v / np.linalg.norm(v))
PROTO_MAT = np.stack(P)
DOMAINS = [l for l in PROTO_LABELS if l != "범위밖"]
SUGGEST_POOL = [(lab, q, i) for i, (lab, q) in enumerate((l, q) for l in PROTO_LABELS for q in PROTO_QUESTIONS[l]) if lab != "범위밖"]

def tag_chunk(i):
    text = chunk_df.embed_text.iat[i]
    hits = kw_hits(text, DOMAIN_KEYWORDS)
    if hits:
        doms = sorted(hits, key=lambda d: -hits[d])
    else:
        sims = PROTO_MAT[[PROTO_LABELS.index(d) for d in DOMAINS]] @ D_DENSE[i]
        doms = [DOMAINS[int(np.argmax(sims))]]
    return doms, sorted(kw_hits(text, EVENT_DICT))

tags = [tag_chunk(i) for i in range(N)]
chunk_df["domains"] = [t[0] for t in tags]
chunk_df["events"] = [t[1] for t in tags]
ov_path = Path("data/chunk_tags_override.csv")
if ov_path.exists():
    ov = pd.read_csv(ov_path, dtype=str, keep_default_na=False).set_index("chunk_id").to_dict("index")
    for col in ("domains", "events"):
        chunk_df[col] = [ov[c][col].split(";") if c in ov and ov[c].get(col) else v
                         for c, v in zip(chunk_df.chunk_id, chunk_df[col])]
    unknown = set(ov) - set(chunk_df.chunk_id)
    print(f"태그 수정 {len(ov) - len(unknown)}건 반영" + (f" | 없는 chunk_id {len(unknown)}개: {sorted(unknown)[:5]}" if unknown else ""))
CH_DOMAINS = [set(x) for x in chunk_df.domains]
CH_EVENTS = [set(x) for x in chunk_df.events]

chunk_df.assign(domains=chunk_df.domains.str.join(";"), events=chunk_df.events.str.join(";"))[
    ["chunk_id", "domains", "events"]].to_csv(Path(CONFIG["RESULTS_DIR"]) / "chunk_tags.csv", index=False, encoding="utf-8-sig")
print("영역 태그 분포:", Counter(d for ds in chunk_df.domains for d in ds))
chunk_df[["chunk_id", "domains", "events"]].head(12)

Inference Embeddings: 100%|██████████| 7/7 [00:13<00:00,  1.93s/it]

영역 태그 분포: Counter({'근태관리': 8, '복리후생': 6, '교육·평가': 5, '업무지원': 5, '급여보상': 3})


,chunk_id,domains,events
0,SAMPLE_누리정밀_경조금지급규정::제1조::0,[복리후생],[]
1,SAMPLE_누리정밀_경조금지급규정::제3조::0,"[복리후생, 근태관리]",[휴직·퇴직]
2,SAMPLE_누리정밀_경조금지급규정::제4조::0,[복리후생],"[결혼, 출산]"
3,SAMPLE_누리정밀_경조금지급규정::제5조::0,[복리후생],"[결혼, 출산]"
4,SAMPLE_누리정밀_경조금지급규정::제6조::0,[복리후생],[]
5,SAMPLE_누리정밀_교육훈련규정::제3조::0,[교육·평가],[입사·승진]
6,SAMPLE_누리정밀_교육훈련규정::제10조::0,[교육·평가],[]
7,SAMPLE_누리정밀_교육훈련규정::제11조::0,[교육·평가],[휴직·퇴직]
8,SAMPLE_누리정밀_여비규정::제2조::0,[업무지원],[]
9,SAMPLE_누리정밀_여비규정::제8조::0,[업무지원],[]


## 4. 평가셋 불러오기
A0와 같은 규칙으로 정답을 읽습니다. `history`, `expected_route`는 없어도 됩니다.

In [37]:
REQUIRED = ["qid", "question", "intent", "domain", "event_type", "target_doc", "target_article", "context_type", "needs_tool"]
eval_df = pd.read_csv(CONFIG["EVAL_PATH"], dtype=str, keep_default_na=False)
missing_cols = [c for c in REQUIRED if c not in eval_df.columns]
assert not missing_cols, f"평가셋에 컬럼이 없습니다: {missing_cols}"
assert eval_df.qid.is_unique, "qid가 중복됩니다"
for c in ("history", "expected_route"):
    if c not in eval_df.columns:
        eval_df[c] = ""

_key = lambda s: re.sub(r"\s+", "", str(s))
doc_lookup = {_key(t): t for t in chunk_df.doc_title.unique()}
valid_pairs = set(zip(chunk_df.doc_title, chunk_df.article))

def parse_gold(row):
    docs_ = [d.strip() for d in row.target_doc.split(";") if d.strip()]
    arts = [norm_article(a) for a in row.target_article.split(";") if a.strip()]
    if not arts:
        return []
    if len(docs_) == 1:
        docs_ = docs_ * len(arts)
    if len(docs_) != len(arts):
        raise ValueError(f"{row.qid}: target_doc {len(docs_)}개 / target_article {len(arts)}개 개수가 맞지 않습니다")
    return [(doc_lookup.get(_key(d), d), a) for d, a in zip(docs_, arts)]

ROUTE_BY_INTENT = {"민감": "담당자연결", "내정보": "개인조회", "범위밖": "범위밖"}
eval_df["gold"] = eval_df.apply(parse_gold, axis=1)
eval_df["turns"] = eval_df.history.apply(lambda s: [h.strip() for h in s.split("||") if h.strip()])
eval_df["exp_route"] = [r.expected_route or ROUTE_BY_INTENT.get(r.intent, "검색") for r in eval_df.itertuples()]
bad = [(r.qid, g) for r in eval_df.itertuples() for g in r.gold if g not in valid_pairs]
if bad:
    print("⚠️ 청크 목록에 없는 정답 (오타 또는 청킹 오류) — 미검색으로 처리됩니다:")
    for q, g in bad:
        print("  ", q, g)

in_scope_eval = eval_df.intent.isin(CONFIG["EVAL_INTENTS"])
answerable = eval_df[in_scope_eval & (eval_df.gold.str.len() > 0)].copy()
no_answer = eval_df[in_scope_eval & (eval_df.gold.str.len() == 0)].copy()
print(f"전체 {len(eval_df)} | 검색 평가 {len(answerable)} | 문서에 없음 {len(no_answer)} | 멀티턴 {int((eval_df.turns.str.len() > 0).sum())}")
print("정답 라우트 분포:", dict(Counter(eval_df.exp_route)))
unk = sorted(set(eval_df.domain) - set(DOMAINS) - {""})
if unk:
    print(f"⚠️ 평가셋 domain 중 사전에 없는 영역: {unk} → DOMAIN_KEYWORDS·PROTO_QUESTIONS 키 이름을 맞춰 주세요")

전체 27 | 검색 평가 21 | 문서에 없음 2 | 멀티턴 2
정답 라우트 분포: {'검색': 22, '개인조회': 1, '범위밖': 2, '담당자연결': 1, '되묻기': 1}


## 5. ① 질문 재작성 (규칙)
1. **줄임말 치환**: `ABBR_DICT` ("육휴" → "육아휴직")
2. **멀티턴 이어받기**: 질문에 지시어("그럼", "그건")가 있거나 업무 영역 키워드가 없으면, 직전 발화의 이벤트·조건·주제어를 앞에 붙입니다.
   - "와이프가 다음 달에 출산 예정이에요" → "그럼 저는 휴가 며칠 받아요?" ⇒ `출산 배우자 그럼 저는 휴가 며칠 받아요? 휴가`
3. **복합 질문 분리**: "~이고 / ~랑 / 그리고 / ?"로 나눴을 때 **조각마다 업무 영역 키워드가 있고 영역이 서로 다를 때만** 나누고, 이벤트·조건 단어는 조각마다 공유합니다. 조각마다 따로 검색해서 결과를 번갈아 합칩니다.
4. **규정 용어 덧붙이기**: `TERM_DICT`와 조건 표준값(와이프 → 배우자)을 뒤에 붙입니다. 원문을 지우지 않으니 dense 검색이 망가지지 않습니다.

재작성은 사전에 있는 말만 붙이고, **사전에 없는 정보를 지어내지 않습니다.**

In [38]:
def normalize_q(q):
    q = re.sub(r"\s+", " ", str(q)).strip()
    for k in sorted(ABBR_DICT, key=len, reverse=True):
        q = q.replace(k, ABBR_DICT[k])
    return q

def extract_slots(text):
    return {slot: val for slot, vals in SLOT_DICT.items() for val, words in vals.items() if any(w in text for w in words)}

def topic_words(text):
    return [w for words in DOMAIN_KEYWORDS.values() for w in words if w in text]

def event_words(text):
    """이벤트 라벨("휴직·퇴직")이 아니라 질문에 실제로 나온 단어("퇴사")를 돌려줌"""
    return [w for words in EVENT_DICT.values() for w in words if w in text and len(w) >= 2]

def expansions(text):
    adds = [v for k, v in TERM_DICT.items() if k in text] + list(extract_slots(text).values())
    out = []
    for a in adds:
        if a not in text and a not in out:
            out.append(a)
    return out

SPLIT_RE = re.compile(r"(?<=[가-힣])(?:이고|하고|이랑|랑|고)\s+|\s*그리고\s+|[?？]\s+")

def split_compound(q):
    parts = [p.strip() for p in SPLIT_RE.split(q) if p and p.strip()]
    merged, topic = [], []
    for p in parts:
        has_topic = bool(topic_words(p))
        if merged and (not has_topic or not topic[-1]):
            merged[-1] += " " + p                      # 주제어 없는 조각은 이웃과 합침
            topic[-1] = topic[-1] or has_topic
        else:
            merged.append(p); topic.append(has_topic)
    doms = [set(kw_hits(p, DOMAIN_KEYWORDS)) for p in merged]
    if len(merged) < 2 or all(d == doms[0] for d in doms):  # 영역이 같으면 한 번에 검색하는 게 나음
        return [q]
    return merged

def rewrite(question, history=None):
    q = normalize_q(question)
    carried = []
    if history:
        prev = normalize_q(history[-1])
        if any(a in q for a in ANAPHORA) or not topic_words(q):
            carried = event_words(prev) + list(extract_slots(prev).values()) + topic_words(prev)
            carried = [c for i, c in enumerate(carried) if c not in q and c not in carried[:i]]
    base = " ".join(carried + [q])
    shared = event_words(base) + list(extract_slots(base).values())
    subs = []
    for part in split_compound(base):
        add = [s for s in shared if s not in part]
        p = " ".join(add + [part])
        subs.append(" ".join([p] + expansions(p)))
    full = " ".join([base] + expansions(base))
    return {"original": question, "normalized": q, "carried": carried, "rewritten": full, "subqueries": subs}

for q, h in [("육휴 신청은 며칠 전까지 해야 돼?", None),
             ("그럼 저는 휴가 며칠 받아요?", ["와이프가 다음 달에 출산 예정이에요"]),
             ("결혼하면 휴가 며칠이고 경조금은 얼마예요?", None),
             ("지원받고 바로 퇴사하면 어떻게 돼요?", ["대학원 다니려고 하는데요"])]:
    r = rewrite(q, h)
    print(f"{q}\n   → {r['rewritten']}\n   → 검색 질의 {r['subqueries']}\n")

육휴 신청은 며칠 전까지 해야 돼?
   → 육아휴직 신청은 며칠 전까지 해야 돼?
   → 검색 질의 ['육아휴직 신청은 며칠 전까지 해야 돼?']

그럼 저는 휴가 며칠 받아요?
   → 출산 배우자 그럼 저는 휴가 며칠 받아요?
   → 검색 질의 ['출산 배우자 그럼 저는 휴가 며칠 받아요?']

결혼하면 휴가 며칠이고 경조금은 얼마예요?
   → 결혼하면 휴가 며칠이고 경조금은 얼마예요?
   → 검색 질의 ['결혼하면 휴가 며칠', '결혼 경조금은 얼마예요?']

지원받고 바로 퇴사하면 어떻게 돼요?
   → 대학원 지원받고 바로 퇴사하면 어떻게 돼요? 퇴직
   → 검색 질의 ['대학원 지원받고 바로 퇴사하면 어떻게 돼요? 퇴직']



## 6. ② 질문 분류 (BGE-M3 프로토타입 + 규칙) · ③ 라우터
| 항목 | 판단 방법 |
|---|---|
| 업무 영역 `domains` (멀티라벨) | 키워드가 있으면 키워드 영역(많이 나온 순) + 프로토타입 1등(유사도가 `high_conf` 이상이면). 키워드가 없으면 프로토타입 1등과 `PROTO_MARGIN` 안에 있는 영역 전부 |
| `confidence` | **high**: 키워드 1·2등 안에 프로토타입 1등이 있음, 또는 키워드 없이 프로토타입 유사도 ≥ `high_conf`이고 2등과 차이가 큼 / 나머지 **low**. LLM이 스스로 매긴 점수가 아니라 두 방법이 **일치하는지**로 정합니다 |
| 범위 안 `in_scope` | 영역·이벤트 키워드가 있거나, 프로토타입 1등이 `in_scope` 임계값 이상이고 "범위밖" 프로토타입보다 가까움 |
| 개인 조회 / 둘다 | 1인칭 + 조회 표현 → 개인조회. 여기에 규정 표현(수당, 언제까지 등)도 있으면 둘다 → 검색하고 개인 기록은 HR 시스템 안내 |
| 애매함 (2가지) | **조건 애매**: `CLARIFY_RULES` (경조금인데 무슨 경조사인지 없음) / **주제 애매**: 키워드 없음 + 1·2등 차이가 작고 유사도도 낮음 |

라우터 순서: **민감 → 담당자연결 / 범위 밖 → 거절 / 개인 조회(둘다 아님) → HR 시스템 / 애매 → 되묻기 / 나머지 → 검색**

In [39]:
def classify(rw):
    text, orig = rw["rewritten"], normalize_q(rw["original"])
    qv = encode_query(text)["dense"]
    sims = dict(zip(PROTO_LABELS, (PROTO_MAT @ qv).tolist()))
    oos = sims.pop("범위밖")
    ranked = sorted(sims.items(), key=lambda x: -x[1])
    best_d, best_s = ranked[0]
    margin = best_s - ranked[1][1]
    kw = kw_hits(text, DOMAIN_KEYWORDS)
    kw_domains = sorted(kw, key=lambda d: -kw[d])
    events = sorted(kw_hits(text, EVENT_DICT))
    if kw_domains:
        domains = kw_domains + ([best_d] if best_d not in kw_domains and best_s >= TH["high_conf"] else [])
        confidence = "high" if best_d in kw_domains[:2] else "low"
    else:
        domains = [d for d, s in ranked if s >= best_s - CONFIG["PROTO_MARGIN"]]
        confidence = "high" if best_s >= TH["high_conf"] and margin >= CONFIG["PROTO_MARGIN"] else "low"

    personal = bool(FIRST_PERSON_RE.search(orig)) and any(w in orig for w in LOOKUP_WORDS)
    both = personal and any(w in orig for w in REG_CUES)
    sensitive = any(w in orig for w in SENSITIVE_WORDS)
    in_scope = bool(kw_domains or events) or (best_s >= TH["in_scope"] and best_s > oos)

    ambiguous, ask = None, None
    for rule in CLARIFY_RULES:
        if any(w in text for w in rule["if_any"]) and not any(w in text for w in rule["unless_any"]):
            ambiguous, ask = "조건", rule["ask"]
            break
    if not ambiguous and in_scope and not kw_domains and not events \
            and best_s < TH["high_conf"] and margin < CONFIG["PROTO_MARGIN"]:
        ambiguous = "주제"
        ask = f"어떤 내용이 궁금하신지 조금 더 알려주시겠어요? 예: {ranked[0][0]} 관련인지, {ranked[1][0]} 관련인지"
    return {"domains": domains, "events": events, "slots": extract_slots(text), "confidence": confidence,
            "in_scope": in_scope, "personal": personal, "both": both, "sensitive": sensitive,
            "ambiguous": ambiguous, "ask": ask, "best_domain": best_d, "best_sim": round(best_s, 4),
            "margin": round(margin, 4), "oos_sim": round(oos, 4), "kw_domains": kw_domains}

def route(c):
    if c["sensitive"]:
        return "담당자연결"
    if not c["in_scope"]:
        return "범위밖"
    if c["personal"] and not c["both"]:
        return "개인조회"
    if c["ambiguous"]:
        return "되묻기"
    return "검색"

for q in ["제 연차 몇 개 남았어요?", "오늘 점심 메뉴 추천해줘", "경조금 얼마 나와요?", "국내 출장 가면 일비 얼마 나와요?"]:
    c = classify(rewrite(q))
    print(f"{q:28s} → {route(c):5s} | 영역 {c['domains']} ({c['confidence']}) | 이벤트 {c['events']} | 1등 {c['best_domain']} {c['best_sim']:.2f}")

제 연차 몇 개 남았어요?               → 개인조회  | 영역 ['근태관리'] (high) | 이벤트 [] | 1등 근태관리 0.71
오늘 점심 메뉴 추천해줘                → 범위밖   | 영역 ['근태관리', '복리후생', '급여보상'] (low) | 이벤트 [] | 1등 근태관리 0.57
경조금 얼마 나와요?                  → 되묻기   | 영역 ['복리후생'] (high) | 이벤트 [] | 1등 복리후생 0.76
국내 출장 가면 일비 얼마 나와요?          → 검색    | 영역 ['업무지원'] (high) | 이벤트 [] | 1등 업무지원 0.85


## 7. ④ 검색기
1. **후보 뽑기**: 전체 청크에 대해 dense 점수와 sparse 점수를 계산 → 각각 상위 `N_CANDIDATES`
2. **필터** (구성 B·C)
   - `hard`: 질문 영역·이벤트와 태그가 하나라도 겹치는 청크만 남김
   - `adaptive` (구성 C): confidence **high**면 하드 필터, **low**면 소프트(겹치는 청크의 RRF 점수만 `SOFT_BOOST`만큼 올림). 하드 필터 뒤 결과가 `MIN_FILTERED`보다 적거나 1등 점수가 `answer` 임계값보다 낮으면 **필터를 풀고 다시 검색**해서 더 나은 쪽을 씁니다.
3. **RRF**: 점수 단위가 다른 두 검색기를 순위로 합칩니다. `1 / (60 + 순위)`의 합
4. **ColBERT 재정렬**: RRF 상위 `RERANK_POOL`개를 `0.4·dense + 0.2·sparse + 0.4·colbert`로 다시 정렬
5. **조항 단위로 합치기**: 같은 조항의 조각은 가장 높은 것 하나만 남김 → Top-K

In [40]:
W = CONFIG["M3_WEIGHTS"]

def sparse_scores(qs):
    s = np.zeros(N, dtype=np.float32)
    for t, w in qs.items():
        for i, dw in SPARSE_INV.get(t, ()):
            s[i] += w * dw
    return s

# (선택) BM25 + Kiwi — Sparse 비교용 (구성 A1k)
BM25 = None
try:
    from kiwipiepy import Kiwi
    _kiwi = Kiwi()
    for w in {x for ws in DOMAIN_KEYWORDS.values() for x in ws if " " not in x and len(x) >= 2}:
        _kiwi.add_user_word(w, "NNG")          # 규정 핵심 용어가 쪼개지지 않게 (첫 tokenize 전에 등록)
    KEEP_TAGS = ("NNG", "NNP", "VV", "VA", "XR", "SL", "SN", "MAG")
    def kiwi_tokens(t):
        return [tok.form for tok in _kiwi.tokenize(t) if tok.tag.startswith(KEEP_TAGS)]

    class BM25Index:
        def __init__(self, docs, k1=1.5, b=0.75):
            self.k1, self.b = k1, b
            self.dl = np.array([len(d) for d in docs], dtype=np.float32)
            self.avgdl = float(self.dl.mean())
            df = Counter(t for d in docs for t in set(d))
            self.idf = {t: float(np.log(1 + (len(docs) - n + 0.5) / (n + 0.5))) for t, n in df.items()}
            self.inv = defaultdict(list)
            for i, d in enumerate(docs):
                for t, f in Counter(d).items():
                    self.inv[t].append((i, f))
        def scores(self, q_tokens):
            s = np.zeros(len(self.dl), dtype=np.float32)
            for t in set(q_tokens):
                for i, f in self.inv.get(t, ()):
                    s[i] += self.idf[t] * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * self.dl[i] / self.avgdl))
            return s
    BM25 = BM25Index([kiwi_tokens(t) for t in chunk_df.embed_text])
    print("BM25 + Kiwi 인덱스 준비 완료")
except ImportError:
    print("kiwipiepy가 없어 BM25(A1k) 비교는 건너뜁니다.")

def rrf_fuse(score_lists, allowed=None):
    fused = np.zeros(N, dtype=np.float64)
    for s in score_lists:
        s = s.astype(np.float64).copy()
        if allowed is not None:
            s[~allowed] = -np.inf
        top = np.argsort(-s)[:CONFIG["N_CANDIDATES"]]
        top = top[np.isfinite(s[top]) & (s[top] > 0)]
        fused[top] += 1.0 / (CONFIG["RRF_K"] + np.arange(1, len(top) + 1))
    return fused

def colbert_score(qc, i):
    return float((qc @ D_COLBERT[i].astype(np.float32).T).max(1).mean())

def to_articles(chunk_hits, k):
    seen, out = set(), []
    for h in chunk_hits:
        key = (h["doc_title"], h["article"])
        if key not in seen:
            seen.add(key); out.append({**h, "key": key})
        if len(out) >= k:
            break
    return out

def retrieve_one(text, cfg, cls=None, k=CONFIG["TOP_K"]):
    qe = encode_query(text)
    t0 = time.perf_counter()
    dense = D_DENSE @ qe["dense"]
    sp = sparse_scores(qe["sparse"]) if (cfg["sparse"] == "m3" or cfg["colbert"]) else None
    lists = [dense]
    if cfg["sparse"] == "m3":
        lists.append(sp)
    elif cfg["sparse"] == "bm25":
        lists.append(BM25.scores(kiwi_tokens(text)))

    mode, filt, allowed, match = cfg["filter"], "none", None, None
    if mode != "none" and cls and (cls["domains"] or cls["events"]):
        qd, qev = set(cls["domains"]), set(cls["events"])
        match = np.array([bool(cd & qd) or bool(ce & qev) for cd, ce in zip(CH_DOMAINS, CH_EVENTS)])
        if mode == "hard" or (mode == "adaptive" and cls["confidence"] == "high"):
            allowed, filt = match, "hard"
        else:
            filt = "soft"
    fused = rrf_fuse(lists, allowed)
    if filt == "soft":
        fused = fused * np.where(match, 1 + CONFIG["SOFT_BOOST"], 1.0)
    order = np.argsort(-fused)
    order = order[fused[order] > 0]

    if cfg["colbert"]:
        pool = order[:CONFIG["RERANK_POOL"]]
        sc = np.array([W["dense"] * dense[i] + W["sparse"] * sp[i] + W["colbert"] * colbert_score(qe["colbert"], i) for i in pool])
        o = np.argsort(-sc)
        order, scores = pool[o], sc[o]
    else:
        order = order[:CONFIG["N_CANDIDATES"]]
        scores = dense[order]               # 재정렬이 없을 때 판단용 점수는 dense 유사도
    hits = [{"idx": int(i), "chunk_id": chunk_df.chunk_id.iat[i], "doc_title": chunk_df.doc_title.iat[i],
             "article": chunk_df.article.iat[i], "score": float(s), "dense": float(dense[i])} for i, s in zip(order, scores)]
    out = {"hits": to_articles(hits, k), "filter": filt, "ms": (time.perf_counter() - t0) * 1000}

    if mode == "adaptive" and filt == "hard":
        weak = len(out["hits"]) < CONFIG["MIN_FILTERED"] or out["hits"][0]["score"] < TH["answer"]
        if weak:
            alt = retrieve_one(text, {**cfg, "filter": "none"}, None, k)
            if alt["hits"] and (not out["hits"] or alt["hits"][0]["score"] > out["hits"][0]["score"]):
                return {**alt, "filter": "hard→해제", "ms": out["ms"] + alt["ms"]}
    return out

def retrieve(rw, cls, cfg, k=CONFIG["TOP_K"]):
    texts = rw["subqueries"] if cfg["rewrite"] else [rw["original"]]
    res = [retrieve_one(t, cfg, cls if cfg["filter"] != "none" else None, k) for t in texts]
    if len(res) == 1:
        return {**res[0], "n_sub": 1}
    merged, seen = [], set()                         # 복합 질문: 조각별 결과를 번갈아 합침
    for rank in range(k):
        for r in res:
            if rank < len(r["hits"]) and r["hits"][rank]["key"] not in seen:
                seen.add(r["hits"][rank]["key"]); merged.append(r["hits"][rank])
    return {"hits": merged[:k], "filter": "/".join(r["filter"] for r in res), "ms": sum(r["ms"] for r in res), "n_sub": len(res)}

CONFIGS = {
    "A0":  dict(sparse=None,   colbert=False, rewrite=False, filter="none"),
    "A1":  dict(sparse="m3",   colbert=False, rewrite=False, filter="none"),
    "A1k": dict(sparse="bm25", colbert=False, rewrite=False, filter="none"),
    "A2":  dict(sparse="m3",   colbert=True,  rewrite=False, filter="none"),
    "A3":  dict(sparse="m3",   colbert=True,  rewrite=True,  filter="none"),
    "B":   dict(sparse="m3",   colbert=True,  rewrite=True,  filter="hard"),
    "C":   dict(sparse="m3",   colbert=True,  rewrite=True,  filter="adaptive"),
}
if BM25 is None:
    CONFIGS.pop("A1k")
CONFIG_DESC = {"A0": "Dense만", "A1": "+ Sparse(BGE-M3) RRF", "A1k": "Dense + BM25(Kiwi) RRF",
               "A2": "+ ColBERT 재정렬", "A3": "+ 질문 재작성", "B": "+ 하드 필터", "C": "+ 적응형 필터·fallback (최종)"}
FINAL = CONFIGS["C"]

_rw = rewrite("국내 출장 가면 일비 얼마 나와요?"); _c = classify(_rw)
for h in retrieve(_rw, _c, FINAL)["hits"]:
    print(f"{h['score']:.3f}  {h['doc_title']} {h['article']}")

BM25 + Kiwi 인덱스 준비 완료
0.636  SAMPLE_누리정밀_여비규정 제8조
0.612  SAMPLE_누리정밀_여비규정 제9조
0.545  SAMPLE_누리정밀_여비규정 제2조
0.447  SAMPLE_누리정밀_여비규정 제13조
0.444  SAMPLE_누리정밀_여비규정 제12조


## 8. 실패 처리 · 임계값 보정
**실패 처리**: 1등 점수가 `answer` 임계값보다 낮으면
1. 분류된 영역의 대표 키워드 3개와 이벤트 단어를 붙여 **확장 재검색 1회** — 단, 통과 여부는 키워드를 붙인 질의가 아니라 **원래 질문 기준 점수**로 판단합니다 (붙인 키워드 때문에 점수가 부풀어 "문서에 없음" 질문이 새어 나가는 것을 막기 위해)
2. 그래도 낮으면 **답변 불가** + 대표 질문 중 가장 가까운 3개를 **추천 질문**으로 제시

**임계값 보정** (`AUTO_CALIBRATE=True`)
- `answer`: "답 있음" 질문과 "문서에 없음" 질문의 1등 점수를 가장 잘 가르는 값 (둘 다 2개 이상 있을 때)
- `in_scope`: 키워드가 없는 질문 중 범위 안 / 범위 밖의 프로토타입 유사도를 가장 잘 가르는 값 (범위 밖이 2개 이상일 때)

⚠️ 같은 평가셋으로 임계값을 정하고 그 평가셋으로 채점하면 **라우팅·답변불가 점수가 약간 부풀려집니다.** 평가셋이 100개를 넘으면 dev/test로 나눠서 dev로만 보정하세요. 검색 지표(Hit·Recall·MRR)는 임계값 영향이 거의 없습니다.

In [41]:
def pair_score(text, idx):
    """질문 text와 청크 idx의 BGE-M3 가중합 점수 (재정렬과 같은 식)"""
    qe = encode_query(text)
    sp = sum(w * D_SPARSE[idx].get(t, 0.0) for t, w in qe["sparse"].items())
    return W["dense"] * float(D_DENSE[idx] @ qe["dense"]) + W["sparse"] * sp + W["colbert"] * colbert_score(qe["colbert"], idx)

def retrieve_with_recovery(rw, cls, cfg=FINAL):
    r = retrieve(rw, cls, cfg)
    if r["hits"] and r["hits"][0]["score"] >= TH["answer"]:
        return r, "ok"
    extra = (DOMAIN_KEYWORDS[cls["domains"][0]][:3] if cls["domains"] else []) + event_words(rw["rewritten"])
    extra = [e for e in dict.fromkeys(extra) if e not in rw["rewritten"]]
    if extra:
        rw2 = {**rw, "subqueries": [s + " " + " ".join(extra) for s in rw["subqueries"]]}
        r2 = retrieve(rw2, cls, cfg)
        # 확장 질의는 키워드를 붙여서 점수가 부풀므로, 통과 여부는 "원래 질문" 기준 점수로 판단
        if r2["hits"] and max(pair_score(s, r2["hits"][0]["idx"]) for s in rw["subqueries"]) >= TH["answer"]:
            return r2, "확장재검색"
    return r, "답변불가"

def suggest_questions(text, n=3):
    sims = PROTO_VECS @ encode_query(text)["dense"]
    pool = sorted(SUGGEST_POOL, key=lambda x: -sims[x[2]])
    return [q for _, q, _ in pool[:n]]

def best_threshold(pos, neg):
    """pos(통과해야 함)와 neg(걸러야 함)를 가장 잘 가르는 값 (balanced accuracy 최대, 동점이면 가운데)"""
    pos, neg = np.asarray(pos, float), np.asarray(neg, float)
    vals = np.unique(np.concatenate([pos, neg]))
    cands = (vals[:-1] + vals[1:]) / 2 if len(vals) > 1 else vals
    accs = np.array([((pos >= t).mean() + (neg < t).mean()) / 2 for t in cands])
    best = np.flatnonzero(accs == accs.max())
    return float(cands[best[len(best) // 2]]), float(accs.max())

# 평가셋 질문마다 재작성·분류 결과를 한 번만 계산해 둡니다
PREP = {}
for r in eval_df.itertuples():
    rw = rewrite(r.question, r.turns or None)
    PREP[r.qid] = (rw, classify(rw))

calib_log = {}
if CONFIG["AUTO_CALIBRATE"]:
    th_before = dict(TH)
    # in_scope: 키워드가 없는 질문만 (키워드가 있으면 임계값과 상관없이 범위 안)
    nokw = [(r, PREP[r.qid][1]) for r in eval_df.itertuples() if not (PREP[r.qid][1]["kw_domains"] or PREP[r.qid][1]["events"])]
    pos = [c["best_sim"] for r, c in nokw if r.exp_route != "범위밖" and c["best_sim"] > c["oos_sim"]]
    neg = [c["best_sim"] for r, c in nokw if r.exp_route == "범위밖" and c["best_sim"] > c["oos_sim"]]
    oos_all = int((eval_df.exp_route == "범위밖").sum())
    if oos_all >= 2 and pos and neg:
        TH["in_scope"], acc = best_threshold(pos, neg); calib_log["in_scope"] = f"{acc:.2f}"
    # answer: 최종 구성에서 1등 점수
    top1 = lambda df: [(retrieve(*PREP[q], FINAL)["hits"] or [{"score": 0}])[0]["score"] for q in df.qid]
    if len(answerable) >= 2 and len(no_answer) >= 2:
        TH["answer"], acc = best_threshold(top1(answerable), top1(no_answer)); calib_log["answer"] = f"{acc:.2f}"
    for k in TH:
        flag = " (보정)" if k in calib_log else " (기본값 유지: 데이터 부족)" if k != "high_conf" else ""
        print(f"{k:9s} {th_before[k]:.3f} → {TH[k]:.3f}{flag}")
    if calib_log:   # 임계값이 바뀌었으니 분류 다시 계산
        PREP = {q: (rw, classify(rw)) for q, (rw, _) in PREP.items()}

in_scope  0.400 → 0.400 (기본값 유지: 데이터 부족)
high_conf 0.550 → 0.550
answer    0.400 → 0.429 (보정)


## 9. ⑤ LLM 답변 생성 · 전체 파이프라인 데모
LLM은 **여기서만** 씁니다. OpenAI 호환 형식으로 넣어 두었고, 환경변수 `LLM_BASE_URL`, `LLM_API_KEY`, `LLM_MODEL`이 없으면 **DRY RUN**으로 프롬프트만 만들어 둡니다. 쓰는 LLM의 호출 방식이 다르면 `call_llm()`만 바꾸면 됩니다.
API 키는 코드에 직접 쓰지 말고 환경변수(Colab이면 보안 비밀)로 넣으세요.

In [42]:
LLM = {"BASE_URL": os.environ.get("LLM_BASE_URL", ""), "API_KEY": os.environ.get("LLM_API_KEY", ""),
       "MODEL": os.environ.get("LLM_MODEL", ""), "TEMPERATURE": 0.0}

SYSTEM_PROMPT = """너는 사내 복리후생·인사 규정 안내 챗봇이다.
1. 아래 [근거]에 있는 내용으로만 답한다. 근거에 없으면 "규정에서 찾을 수 없습니다. 인사팀에 문의해 주세요."라고 답한다.
2. 답의 근거가 된 문장 뒤에 [1 제26조]처럼 근거 번호와 조항을 붙인다.
3. 대상·직급·지역 등 조건에 따라 답이 다르면 조건별로 나눠 답한다.
4. 금액·일수·기한은 근거에 적힌 숫자를 그대로 쓰고, 계산하거나 추측하지 않는다."""

def build_messages(question, hits, cls, history=None):
    ctx = "\n\n".join(f"[{i}] {h['doc_title']} {h['article']}\n{chunk_df.text.iat[h['idx']]}" for i, h in enumerate(hits, 1))
    user = f"[근거]\n{ctx}\n\n"
    if history:
        user += "[이전 대화]\n" + "\n".join(f"- {h}" for h in history[-2:]) + "\n\n"
    user += f"[질문]\n{question}"
    if cls["slots"]:
        user += "\n\n[질문에서 확인된 조건] " + ", ".join(f"{k}={v}" for k, v in cls["slots"].items())
    if cls["both"]:
        user += "\n\n개인 기록(잔여일수·지급 내역)은 답하지 말고 '인사시스템에서 확인할 수 있다'고만 안내한다."
    return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}]

def call_llm(messages):
    if not (LLM["BASE_URL"] and LLM["MODEL"]):
        return None                                   # DRY RUN
    from openai import OpenAI
    client = OpenAI(base_url=LLM["BASE_URL"], api_key=LLM["API_KEY"])
    r = client.chat.completions.create(model=LLM["MODEL"], messages=messages, temperature=LLM["TEMPERATURE"])
    return r.choices[0].message.content

def chat(question, history=None, verbose=True):
    t0 = time.perf_counter()
    rw = rewrite(question, history)
    c = classify(rw)
    rt = route(c)
    out = {"question": question, "rewritten": rw["rewritten"], "subqueries": rw["subqueries"], "route": rt,
           "class": c, "hits": [], "status": "", "reply": ""}
    if rt == "검색":
        r, status = retrieve_with_recovery(rw, c)
        out.update(hits=r["hits"], status=status, filter=r["filter"])
        if status == "답변불가":
            out["reply"] = MSG["답변불가"] + "\n혹시 이런 질문이었나요?\n" + "\n".join(f"  - {s}" for s in suggest_questions(rw["rewritten"]))
        else:
            msgs = build_messages(question, r["hits"], c, history)
            out["messages"] = msgs
            out["reply"] = call_llm(msgs) or f"(DRY RUN) 근거 {len(r['hits'])}개로 LLM 호출 준비 완료"
            if c["both"]:
                out["reply"] += "\n" + MSG["개인조회"]
    elif rt == "되묻기":
        out["reply"] = c["ask"]
    else:
        out["reply"] = MSG[rt]
    out["ms"] = (time.perf_counter() - t0) * 1000
    if verbose:
        print(f"Q: {question}" + (f"   (이전: {history[-1]})" if history else ""))
        print(f"  재작성 : {rw['rewritten']}" + (f"  | 분리 {len(rw['subqueries'])}개" if len(rw["subqueries"]) > 1 else ""))
        print(f"  분류   : 영역 {c['domains']} ({c['confidence']}) | 이벤트 {c['events']} | 조건 {c['slots']}")
        print(f"  라우트 : {rt}" + (f" → {out['status']} (필터 {out.get('filter')})" if rt == "검색" else ""))
        for h in out["hits"][:3]:
            print(f"     {h['score']:.3f}  {h['doc_title']} {h['article']}")
        print("  답변   : " + out["reply"].replace("\n", "\n           ") + "\n")
    return out

for q, h in [("반차는 몇 시부터예요?", None),
             ("그럼 저는 휴가 며칠 받아요?", ["와이프가 다음 달에 출산 예정이에요"]),
             ("결혼하면 휴가 며칠이고 경조금은 얼마예요?", None),
             ("제 연차 몇 개 남았는지랑 못 쓰면 수당 나오는지 알려줘", None),
             ("경조금 얼마 나와요?", None),
             ("반려동물 수당 있어요?", None),
             ("오늘 점심 메뉴 추천해줘", None),
             ("팀장님한테 계속 폭언 들어요 어디에 말해야 돼요?", None)]:
    _ = chat(q, h)

Q: 반차는 몇 시부터예요?
  재작성 : 반차는 몇 시부터예요?
  분류   : 영역 ['근태관리'] (high) | 이벤트 [] | 조건 {}
  라우트 : 검색 → ok (필터 hard)
     0.497  SAMPLE_누리정밀_취업규칙 제25조
     0.345  SAMPLE_누리정밀_취업규칙 제25조의2
     0.343  SAMPLE_누리정밀_취업규칙 제26조
  답변   : (DRY RUN) 근거 5개로 LLM 호출 준비 완료

Q: 그럼 저는 휴가 며칠 받아요?   (이전: 와이프가 다음 달에 출산 예정이에요)
  재작성 : 출산 배우자 그럼 저는 휴가 며칠 받아요?
  분류   : 영역 ['근태관리'] (high) | 이벤트 ['출산'] | 조건 {'대상': '배우자'}
  라우트 : 검색 → ok (필터 hard)
     0.586  SAMPLE_누리정밀_취업규칙 제27조
     0.490  SAMPLE_누리정밀_취업규칙 제25조
     0.484  SAMPLE_누리정밀_취업규칙 제26조
  답변   : (DRY RUN) 근거 5개로 LLM 호출 준비 완료

Q: 결혼하면 휴가 며칠이고 경조금은 얼마예요?
  재작성 : 결혼하면 휴가 며칠이고 경조금은 얼마예요?  | 분리 2개
  분류   : 영역 ['근태관리', '복리후생'] (high) | 이벤트 ['결혼'] | 조건 {}
  라우트 : 검색 → ok (필터 hard/hard)
     0.501  SAMPLE_누리정밀_취업규칙 제26조
     0.606  SAMPLE_누리정밀_경조금지급규정 제4조
     0.484  SAMPLE_누리정밀_취업규칙 제27조
  답변   : (DRY RUN) 근거 5개로 LLM 호출 준비 완료

Q: 제 연차 몇 개 남았는지랑 못 쓰면 수당 나오는지 알려줘
  재작성 : 제 연차 몇 개 남았는지랑 못 쓰면 수당 나오는지 알려줘 미사용 수당  | 분리 2개
  분류   : 영역 ['급여보상', '근태관리'] (high) | 이벤트 [] | 조건

In [43]:

# LLM에 실제로 들어가는 프롬프트 확인
demo = chat("아버지 팔순인데 축하금 나와요?", verbose=False)
for m in demo.get("messages", []):
    print(f"── {m['role']} ──\n{m['content']}\n")

── system ──
너는 사내 복리후생·인사 규정 안내 챗봇이다.
1. 아래 [근거]에 있는 내용으로만 답한다. 근거에 없으면 "규정에서 찾을 수 없습니다. 인사팀에 문의해 주세요."라고 답한다.
2. 답의 근거가 된 문장 뒤에 [1 제26조]처럼 근거 번호와 조항을 붙인다.
3. 대상·직급·지역 등 조건에 따라 답이 다르면 조건별로 나눠 답한다.
4. 금액·일수·기한은 근거에 적힌 숫자를 그대로 쓰고, 계산하거나 추측하지 않는다.

── user ──
[근거]
[1] SAMPLE_누리정밀_경조금지급규정 제4조
제4조(지급기준) 경조금은 다음 표에 따라 지급한다.
| 구분 | 대상 | 경조금 | 화환 |
|---|---|---|---|
| 결혼 | 본인 | 500,000원 | 지원 |
| 결혼 | 자녀 | 300,000원 | 지원 |
| 출산 | 본인 또는 배우자 | 300,000원 | - |
| 회갑·칠순·팔순 | 본인 및 배우자의 부모 | 200,000원 | - |
| 사망 | 배우자, 본인 및 배우자의 부모 | 1,000,000원 | 조화 지원 |

[2] SAMPLE_누리정밀_경조금지급규정 제6조
제6조(중복지급의 제한) 부부가 모두 직원인 경우 같은 경조사에 대한 경조금은 1인에게만 지급한다.

[3] SAMPLE_누리정밀_경조금지급규정 제5조
제5조(신청) ① 경조금을 받으려는 직원은 경조사 발생일로부터 60일 이내에 인사시스템으로 신청하여야 한다.
② 신청 시 청첩장, 출생증명서, 가족관계증명서 등 증빙서류를 첨부하여야 한다.

[4] SAMPLE_누리정밀_경조금지급규정 제3조
제3조(지급대상) 경조금은 재직 중인 직원에게 지급한다. 휴직 중인 직원도 지급대상에 포함한다.

[5] SAMPLE_누리정밀_경조금지급규정 제1조
제1조(목적) 이 규정은 직원의 경조사에 대한 경조금 지급 기준과 절차를 정함을 목적으로 한다.

[질문]
아버지 팔순인데 축하금 나와요?

[질문에서 확인된 조건] 대상=부모



## 10. 평가
### 10-1. 라우팅 · 분류
- **라우팅 정확도**: 검색 전에 정한 라우트가 `expected_route`와 같은지
- **영역 정확도**: top-1이 평가셋 `domain`과 같은지 / 붙인 영역 안에 정답이 있는지(멀티라벨 기준)
- **이벤트 정확도**: 찾은 이벤트가 `event_type`과 같은지 (`없음`은 이벤트를 하나도 안 찾아야 정답)
- **답변불가 판단**: "문서에 없음" 질문을 답변불가로 막았는지 / 답 있는 질문을 잘못 막지 않았는지

In [44]:
rows = []
for r in eval_df.itertuples():
    rw, c = PREP[r.qid]
    rt = route(c)
    final = rt
    if rt == "검색" and r.intent in CONFIG["EVAL_INTENTS"]:
        _, status = retrieve_with_recovery(rw, c)
        final = "답변불가" if status == "답변불가" else "답변"
    exp_ev = set() if r.event_type in ("", "없음") else {r.event_type}
    rows.append({"qid": r.qid, "question": r.question, "intent": r.intent, "exp_route": r.exp_route, "route": rt,
                 "route_ok": rt == r.exp_route, "final": final, "has_gold": len(r.gold) > 0,
                 "domain": r.domain, "pred_domains": ";".join(c["domains"]), "confidence": c["confidence"],
                 "dom_top1": (c["domains"][:1] == [r.domain]) if r.domain else np.nan,
                 "dom_any": (r.domain in c["domains"]) if r.domain else np.nan,
                 "event_ok": set(c["events"]) == exp_ev, "pred_events": ";".join(c["events"]),
                 "rewritten": rw["rewritten"], "n_sub": len(rw["subqueries"]), "best_sim": c["best_sim"]})
route_df = pd.DataFrame(rows)

print(f"라우팅 정확도 {route_df.route_ok.mean():.3f} ({int(route_df.route_ok.sum())}/{len(route_df)})")

# pd.crosstab(ImportError 우회): dict와 DataFrame 기반으로 간단하게 직접 교차테이블 집계 후 출력합니다.
ct_data = {}
all_routes = sorted(list(set(route_df.exp_route) | set(route_df.route)))
for r_gold in all_routes:
    ct_data[r_gold] = {}
    for r_pred in all_routes:
        ct_data[r_gold][r_pred] = int(((route_df.exp_route == r_gold) & (route_df.route == r_pred)).sum())

ct_df = pd.DataFrame(ct_data).T # 행: 정답(exp_route), 열: 예측(route)
ct_df.index.name = "정답"
ct_df.columns.name = "예측"
display(ct_df)

dom = route_df[route_df.domain != ""]
print(f"영역 top-1 {dom.dom_top1.mean():.3f} | 영역 포함(멀티라벨) {dom.dom_any.mean():.3f} | "
      f"이벤트 {route_df.event_ok.mean():.3f} | confidence high 비율 {(dom.confidence == 'high').mean():.3f}")
hi = dom[dom.confidence == "high"]
if len(hi):
    print(f"  └ confidence high일 때 영역 포함 {hi.dom_any.mean():.3f} (낮으면 하드 필터가 정답을 지움)")

srch = route_df[route_df.final.isin(["답변", "답변불가"])]
na_ok = (srch[~srch.has_gold].final == "답변불가").mean() if (~srch.has_gold).any() else np.nan
fr = (srch[srch.has_gold].final == "답변불가").mean() if srch.has_gold.any() else np.nan
print(f"문서에 없음 → 답변불가로 막음 {na_ok:.3f} | 답 있는 질문을 잘못 막음 {fr:.3f}")
wrong = route_df[~route_df.route_ok]
if len(wrong):
    print("\n라우팅 틀린 질문:")
    display(wrong[["qid", "question", "exp_route", "route", "pred_domains", "best_sim"]])

라우팅 정확도 1.000 (27/27)


예측,개인조회,검색,담당자연결,되묻기,범위밖
정답,,,,,
개인조회,1,0,0,0,0
검색,0,22,0,0,0
담당자연결,0,0,1,0,0
되묻기,0,0,0,1,0
범위밖,0,0,0,0,2


영역 top-1 0.833 | 영역 포함(멀티라벨) 1.000 | 이벤트 1.000 | confidence high 비율 0.917
  └ confidence high일 때 영역 포함 1.000 (낮으면 하드 필터가 정답을 지움)
문서에 없음 → 답변불가로 막음 1.000 | 답 있는 질문을 잘못 막음 0.000


### 10-2. 검색 비교 실험 (A0 → C)
- 지표는 A0 노트북과 같습니다: **Hit@k**(정답 조항 하나라도 top-k), **Recall@5**(정답 조항 중 top-5에 든 비율, 이벤트 질문에서 중요), **MRR@10**
- **P@5**: top-5 중 정답 조항 비율. 정답이 1개면 최대 0.2라 절댓값보다 **구성 간 차이**(노이즈가 줄었는지)로 봅니다.
- **지연(ms)**: 검색 시간만 (질문 인코딩은 캐시돼서 제외, 아래에 따로 표시). LLM 생성 시간은 포함하지 않습니다.
- A0·A1·A2는 재작성을 안 하므로 멀티턴 질문은 직전 발화 없이 원문만으로 검색합니다. 그래서 A3에서 오르는 폭이 **재작성 효과**입니다.

In [45]:
KS = [1, 3, 5, 10]

def run_eval(df, cfg):
    rows = []
    for r in df.itertuples():
        rw, c = PREP[r.qid]
        res = retrieve(rw, c, cfg, k=CONFIG["EVAL_K"])
        keys = [h["key"] for h in res["hits"]]
        gold = set(r.gold)
        ranks = [i + 1 for i, k in enumerate(keys) if k in gold]
        row = {"qid": r.qid, "question": r.question, "intent": r.intent, "domain": r.domain,
               "event_type": r.event_type, "context_type": r.context_type, "multiturn": bool(r.turns),
               "n_gold": len(gold), "first_rank": ranks[0] if ranks else None, "rr": 1 / ranks[0] if ranks else 0.0,
               "p@5": len(gold & set(keys[:5])) / 5, "top1_score": res["hits"][0]["score"] if res["hits"] else np.nan,
               "filter": res["filter"], "latency_ms": res["ms"],
               "gold": " ; ".join(f"{d} {a}" for d, a in r.gold),
               "top3": " | ".join(f"{d} {a} ({h['score']:.2f})" for h, (d, a) in zip(res["hits"][:3], keys[:3]))}
        for k in KS:
            got = len(gold & set(keys[:k]))
            row[f"hit@{k}"] = float(got > 0)
            row[f"recall@{k}"] = got / len(gold)
        rows.append(row)
    return pd.DataFrame(rows)

results, summary_rows = {}, []
for name, cfg in CONFIGS.items():
    res = run_eval(answerable, cfg)
    results[name] = res
    summary_rows.append({"config": name, "설명": CONFIG_DESC[name],
                         **{f"Hit@{k}": res[f"hit@{k}"].mean() for k in KS},
                         "Recall@5": res["recall@5"].mean(), "MRR@10": res["rr"].mean(), "P@5": res["p@5"].mean(),
                         "멀티턴 Hit@5": res[res.multiturn]["hit@5"].mean() if res.multiturn.any() else np.nan,
                         "지연ms": res.latency_ms.mean()})
summary = pd.DataFrame(summary_rows).set_index("config")
print(f"검색 평가 {len(answerable)}문항 | 질문 인코딩 평균 {np.mean([v['ms'] for v in _QCACHE.values()]):.0f}ms (별도)")
display(summary.round(3))

검색 평가 21문항 | 질문 인코딩 평균 1138ms (별도)


,설명,Hit@1,Hit@3,Hit@5,Hit@10,Recall@5,MRR@10,P@5,멀티턴 Hit@5,지연ms
config,,,,,,,,,,
A0,Dense만,0.810,0.905,1.0,1.0,1.0,0.879,0.219,1.0,0.826
A1,+ Sparse(BGE-M3) RRF,0.857,1.000,1.0,1.0,1.0,0.921,0.219,1.0,0.610
A1k,Dense + BM25(Kiwi) RRF,0.810,0.905,1.0,1.0,1.0,0.873,0.219,1.0,1.334
A2,+ ColBERT 재정렬,0.857,0.905,1.0,1.0,1.0,0.905,0.219,1.0,12.433
A3,+ 질문 재작성,0.810,1.000,1.0,1.0,1.0,0.889,0.219,1.0,13.630
B,+ 하드 필터,0.810,1.000,1.0,1.0,1.0,0.889,0.219,1.0,6.428
C,+ 적응형 필터·fallback (최종),0.810,1.000,1.0,1.0,1.0,0.889,0.219,1.0,6.601


In [46]:

# 최종 구성(C)의 영역·질문 유형별 세부 점수
res_c = results["C"]
metric_cols = ["hit@1", "hit@5", "recall@5", "rr", "p@5"]
def breakdown(col):
    g = res_c.groupby(col)[metric_cols].mean()
    g.insert(0, "n", res_c.groupby(col).size())
    return g.rename(columns={"rr": "MRR"}).round(3)
for col in ["domain", "context_type", "event_type", "filter"]:
    display(breakdown(col))

,n,hit@1,hit@5,recall@5,MRR,p@5
domain,,,,,,
교육·평가,2,1.000,1.0,1.0,1.000,0.200
근태관리,7,0.571,1.0,1.0,0.738,0.200
급여보상,2,0.500,1.0,1.0,0.750,0.200
복리후생,6,1.000,1.0,1.0,1.000,0.267
업무지원,4,1.000,1.0,1.0,1.000,0.200


,n,hit@1,hit@5,recall@5,MRR,p@5
context_type,,,,,,
조문,13,0.923,1.0,1.0,0.962,0.20
표,8,0.625,1.0,1.0,0.771,0.25


,n,hit@1,hit@5,recall@5,MRR,p@5
event_type,,,,,,
결혼,3,0.667,1.0,1.0,0.833,0.333
없음,12,0.917,1.0,1.0,0.958,0.200
입사·승진,1,1.000,1.0,1.0,1.000,0.200
출산,2,0.000,1.0,1.0,0.333,0.200
휴직·퇴직,3,1.000,1.0,1.0,1.000,0.200


,n,hit@1,hit@5,recall@5,MRR,p@5
filter,,,,,,
hard,18,0.833,1.0,1.0,0.898,0.211
hard/hard,2,0.500,1.0,1.0,0.750,0.300
soft,1,1.000,1.0,1.0,1.000,0.200


In [47]:
# 구성 사이에 결과가 바뀐 질문 — 어떤 단계가 무엇을 고쳤고 무엇을 망쳤는지
names = list(CONFIGS)
hit5 = pd.DataFrame({n: results[n].set_index("qid")["hit@5"] for n in names})
hit5["question"] = answerable.set_index("qid").question
changed = hit5[hit5[names].nunique(axis=1) > 1]
print(f"구성에 따라 Hit@5가 바뀐 질문 {len(changed)}개 (1=성공, 0=실패)")
display(changed)

fails = res_c[res_c["hit@5"] == 0]
print(f"\n최종(C) 실패 {len(fails)} / {len(res_c)}")
pd.set_option("display.max_colwidth", 120)
display(fails[["qid", "question", "domain", "filter", "gold", "top3"]])

구성에 따라 Hit@5가 바뀐 질문 0개 (1=성공, 0=실패)


,A0,A1,A1k,A2,A3,B,C,question
qid,,,,,,,,



최종(C) 실패 0 / 21


,qid,question,domain,filter,gold,top3


### 10-3. 답 있는 질문 vs 문서에 없는 질문의 1등 점수
두 분포가 잘 갈라질수록 `answer` 임계값이 믿을 만합니다. 겹치면 임계값 대신 "근거에 없으면 모른다고 답하라"는 LLM 지시에 더 기대야 합니다.

In [48]:
if len(no_answer):
    na = run_eval(no_answer.assign(gold=[[("", "")]] * len(no_answer)), FINAL)
    print(f"답 있음     1등 점수: 중앙값 {res_c.top1_score.median():.3f} (최소 {res_c.top1_score.min():.3f})")
    print(f"문서에 없음 1등 점수: 중앙값 {na.top1_score.median():.3f} (최대 {na.top1_score.max():.3f})")
    print(f"현재 answer 임계값 {TH['answer']:.3f}")
    display(na[["qid", "question", "top1_score", "top3"]])
else:
    print("문서에 없음 질문이 없습니다. 평가셋에 10% 정도 넣는 것을 권장합니다.")

답 있음     1등 점수: 중앙값 0.602 (최소 0.434)
문서에 없음 1등 점수: 중앙값 0.413 (최대 0.424)
현재 answer 임계값 0.429


,qid,question,top1_score,top3
0,S14,반려동물 수당 있어요?,0.402261,SAMPLE_누리정밀_취업규칙 제25조의2 (0.40) | SAMPLE_누리정밀_여비규정 제12조 (0.29) | SAMPLE_누리정밀_취업규칙 제1조 (0.26)
1,S25,구내식당 밥값 지원돼요?,0.424316,SAMPLE_누리정밀_여비규정 제8조 (0.42) | SAMPLE_누리정밀_여비규정 제2조 (0.41) | SAMPLE_누리정밀_교육훈련규정 제10조 (0.41)


## 11. 결과 저장 · 실험 로그
- `results/baseline_C_per_question_*.csv`: 최종 구성의 질문별 결과
- `results/routing_*.csv`: 질문별 재작성·분류·라우팅 결과
- `results/experiments.csv`: A0 노트북과 **같은 파일**에 구성별로 한 줄씩 쌓습니다 (컬럼이 달라도 합쳐서 다시 씁니다)

In [49]:
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
out_dir = Path(CONFIG["RESULTS_DIR"])
res_c.to_csv(out_dir / f"baseline_C_per_question_{run_id}.csv", index=False, encoding="utf-8-sig")
route_df.to_csv(out_dir / f"routing_{run_id}.csv", index=False, encoding="utf-8-sig")

common = {"run_id": run_id, "embedder": enc.name, "device": enc.device, "data": "SAMPLE" if USING_SAMPLE else "real",
          "n_docs": len(docs), "n_chunks": N, "n_eval": len(answerable), "chunk_max_chars": CONFIG["CHUNK_MAX_CHARS"],
          "max_seq_len": CONFIG["MAX_SEQ_LEN"], "index_build_s": round(index_sec, 1), "machine": platform.platform(),
          "th_in_scope": round(TH["in_scope"], 4), "th_answer": round(TH["answer"], 4), "calibrated": ",".join(calib_log) or "no",
          "routing_acc": round(route_df.route_ok.mean(), 4)}
log_rows = [{**common, "config": f"{n}_{CONFIG_DESC[n]}", **{k: round(v, 4) for k, v in s.items() if k != "설명"}}
            for n, s in summary.to_dict("index").items()]
log_path = out_dir / "experiments.csv"
log = pd.concat([pd.read_csv(log_path), pd.DataFrame(log_rows)], ignore_index=True) if log_path.exists() else pd.DataFrame(log_rows)
log.to_csv(log_path, index=False, encoding="utf-8-sig")
print("저장:", log_path)
log.tail(len(log_rows))[["run_id", "config", "data", "Hit@1", "Hit@5", "Recall@5", "MRR@10"]]

저장: results/experiments.csv


,run_id,config,data,Hit@1,Hit@5,Recall@5,MRR@10
7,20261113_014241,A0_Dense만,SAMPLE,0.8095,1.0,1.0,0.8786
8,20261113_014241,A1_+ Sparse(BGE-M3) RRF,SAMPLE,0.8571,1.0,1.0,0.9206
9,20261113_014241,A1k_Dense + BM25(Kiwi) RRF,SAMPLE,0.8095,1.0,1.0,0.8730
10,20261113_014241,A2_+ ColBERT 재정렬,SAMPLE,0.8571,1.0,1.0,0.9048
11,20261113_014241,A3_+ 질문 재작성,SAMPLE,0.8095,1.0,1.0,0.8889
12,20261113_014241,B_+ 하드 필터,SAMPLE,0.8095,1.0,1.0,0.8889
13,20261113_014241,C_+ 적응형 필터·fallback (최종),SAMPLE,0.8095,1.0,1.0,0.8889


## 다음 단계
1. **실제 규정·평가셋으로 실행**: `data/regulations/`에 규정을, `data/eval_set.csv`에 평가셋을 넣고 `BASELINE_EMBEDDER=bge-m3`로 처음부터 실행
2. **사전 맞추기**: 노션 6개 업무 영역 이름으로 `DOMAIN_KEYWORDS`·`PROTO_QUESTIONS` 키를 맞추고, 규정 문서를 보며 용어·대표 질문 채우기
3. **청크 태그 확인**: `results/chunk_tags.csv`에서 틀린 태그를 `data/chunk_tags_override.csv`로 수정 → 하드 필터(B)가 정답을 지우는 일이 줄어듭니다
4. **실패 사례 → 사전 보강**: 10-2의 "구성에 따라 바뀐 질문"과 실패 표를 보고 놓친 표현을 사전에 추가 (평가셋 문장을 그대로 넣지는 않기)
5. 평가셋이 커지면 **dev/test 분리** 후 dev로만 임계값 보정